<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açınız](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/NB01_python_ve_ekg_temelleri.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html)

# Week 01: Python and cardiology data basics

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lab.html) · [PDF notes](Week01_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/NB01_python_and_ecg_basics.ipynb)

</div>

**How to use this Jupyter notebook.** The notebook follows the practice stops of the lecture page. Every stop has three parts, and each part also runs on its own: Python basics, application with ready prompts, and the ready application. **Python basics** teaches the Python that the stop needs, with small examples from the topic of the week; its exercises check themselves. **Prompt and application** gives ready prompts for an artificial intelligence (AI) assistant: The assistant writes the code, you run it, and a check cell tells you whether the result is right. **Ready application** is the complete working Python program of the stop, with notes that explain it.

Go through the three parts in this order. If you already know Python, you may start with the prompts; if you only want to see the result, open the ready application. The weekly project at the end stands on its own as well.

**Start cells.** Every part opens with a start cell. It prepares the course package and the packages of the week, and it repeats the data calls and the code of the earlier parts that the cells of the part use. The part therefore runs without errors even when you start there. If you have run the earlier parts in this session, you do not need to run the start cell; running it again does no harm. A few sections of the ready applications, which the lecture page opens directly, have a start cell of their own.

**Data.** Every cell that reads a data set offers three sources: `"library"` reads the data with a Python library from the site of the data set, `"web"` reads the copy in the GitHub repository of the course, and `"computer"` reads a file that you have downloaded. In Google Colab, choose the source in the box on the right of the cell; in Jupyter, change the line `DATA_SOURCE = ...`. With `"computer"`, Jupyter looks for the file in the folder of this notebook, and Colab opens a window to upload it.

**Finding your place.** The table of contents lists every stop and part. In Colab it opens with the list icon on the left; in Jupyter, with View → Table of Contents. Every part ends with a link back to the lecture page. In Colab, your changes are kept only in a copy saved to Drive: Use File → Save a copy in Drive.

**Where it runs.** The notebook runs in Jupyter (Anaconda) on your own computer and in Google Colab. Every data set here is synthetic or comes from a public research database. Real patient data belong only on a computer of your institution, under its rules, and never in a cloud notebook or in an AI chat.

## 0. Setup

**First use of Jupyter Notebook.** A notebook is made of cells. This cell holds text; the grey cells hold Python code. Click into the code cell below and press Shift+Enter: The cell runs, its result appears below it, and the number in the brackets on its left, such as `[1]`, shows the order of the run. The kernel, the Python process behind the notebook, remembers everything that the cells have run, so the cells run from top to bottom. **Kernel → Restart Kernel and Run All Cells** repeats the whole notebook, and Ctrl+S (Cmd+S on a Mac) saves it as an `.ipynb` file.

**The start cell.** The code cell below prepares the session. It finds the course package `vclinic` in the course folder; if the notebook was opened on its own, for example in Google Colab, it downloads the package from GitHub once. It then loads the packages, chooses the language of the messages, installs the two ECG packages of the week, `wfdb` and `neurokit2`, with pip when they are missing and loads the expected answers of the exercises. Every part of the notebook opens with a start cell that holds the same code, together with the data calls of the part.

In [ ]:
# Start cell of the Jupyter notebook
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-setup)*

## Stop 1: Mini project: The Kaggle Heart Disease data set

*Lecture page: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#stop-1)*

This stop is the mini project of the session: loading the Kaggle Heart Disease data set, the Cleveland data of the UCI Heart Disease collection, with a statistical summary and a visualisation. The registry of a clinic is a table: one row per patient, one column per variable. In this stop, the 303 patients of the data set and the first 500 patients of the virtual clinic are summarised by a mean, a share and a count, each with a clinical question behind it.

**At the end of this stop:** The mean age of the Cleveland patients with coronary artery disease (56.6 years), the share of women among all 303 patients (32.0 percent) and the number of patients with HFrEF in the virtual clinic (47), with a box plot of the ejection fraction by diagnosis.

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 1 · Python basics

*Lecture page: [Variables, lists and loops with medical examples](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#variables-lists-and-loops-with-medical-examples)*

Python is the language of this course. A code cell holds Python instructions, and Shift+Enter runs them. Text after `#` is a comment: Python skips it, and it explains the code to the reader. Every example cell of this part begins with the line `# Python practice: a small example for learning the language`. The examples use the numbers of the case of the week, a 67-year-old woman with a heart rate of 85 beats per minute (bpm) and a QT interval of 391 ms.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 1 · Python basics», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

In [ ]:
# Python practice: a small example for learning the language
heart_rate = 85          # beats per minute (bpm)
qt_ms = 391              # QT interval in milliseconds
rr_s = 60 / heart_rate   # RR interval in seconds
print(rr_s)
print(round(rr_s, 3))

**Python note.** `heart_rate = 85` creates a *variable*: a name that holds a value. The sign `=` assigns; it does not test equality. `/` divides, and `60 / heart_rate` gives the RR interval in seconds. `print()` shows a value, and `round(value, 3)` rounds it to three decimals. Python counts in floating-point numbers, so `0.7058823529411765` is printed in full until it is rounded.

In [ ]:
# Python practice: a small example for learning the language
patient_id = "VC-0317"
print(f"Patient {patient_id}: {heart_rate} bpm, RR {rr_s:.3f} s")

**Python note.** Text in quotes is a *string*. A string that starts with `f` is an *f-string*: Values inside curly braces are put into the text, and `{rr_s:.3f}` writes the number with three decimals.

In [ ]:
# Python practice: a small example for learning the language
troponin = [3.6, 4.1, 3.9]          # serial measurements (ng/L)
print(troponin[0], troponin[-1], len(troponin))
troponin.append(4.0)                # a fourth measurement arrives
print(troponin, "max:", max(troponin))

**Python note.** Square brackets make a *list*, an ordered series of values. Python counts positions from 0, so `troponin[0]` is the first value and `troponin[-1]` the last. `len()` counts the elements, `.append()` adds one at the end, and `max()` returns the largest.

In [ ]:
# Python practice: a small example for learning the language
patient = {"id": "VC-0317", "age": 67, "sex": "F", "hr_bpm": 85, "qt_ms": 391}
print(patient["age"])
patient["lvef"] = 63                # a new field: left ventricular ejection fraction (%)
print(patient)

**Python note.** Curly braces with `key: value` pairs make a *dictionary*, which works like a patient chart: Each field has a label. `patient["age"]` reads a field, and `patient["lvef"] = 63` adds one.

In [ ]:
# Python practice: a small example for learning the language
qtc_ms = 465
limit = 460 if patient["sex"] == "F" else 450
if qtc_ms >= limit:
    print(f"QTc {qtc_ms} ms is at or above the limit of {limit} ms")
else:
    print(f"QTc {qtc_ms} ms is below the limit of {limit} ms")

**Python note.** `if` runs the indented lines below it only when its condition is true, and `else` covers the other case. The indentation (four spaces) marks which lines belong to each branch. `==` tests equality and `>=` means "greater than or equal". The line `limit = 460 if ... else 450` chooses one of two values in a single line. The two limits are the thresholds of a prolonged QTc for women and men in the 2009 recommendations of the American Heart Association (AHA) and its partners.

In [ ]:
# Python practice: a small example for learning the language
clinic = [
    {"id": "VC-0317", "hr": 85, "qt": 391},
    {"id": "VC-0079", "hr": 58, "qt": 404},
    {"id": "VC-0254", "hr": 98, "qt": 343},
]
for p in clinic:
    rr = 60 / p["hr"]
    print(p["id"], round(rr, 3))

**Python note.** `clinic` is a list of dictionaries, one per patient. `for p in clinic:` takes the patients one by one, and the indented lines run once for each of them, like a ward round.

In [ ]:
# Python practice: a small example for learning the language
def qtc_bazett(qt_ms, rr_s):
    """QT interval corrected for heart rate with Bazett's formula (ms)."""
    return qt_ms / rr_s ** 0.5


print(round(qtc_bazett(391, 60 / 85), 1))

**Python note.** `def` defines a *function*, a named procedure that can be used again, much like a clinical protocol. `qt_ms` and `rr_s` are its inputs, and `return` gives back the result. The text in triple quotes is the docstring, the short description of the function. `**` raises to a power, so `rr_s ** 0.5` is the square root.

**On the lecture page:** [Worked example: From RR intervals to heart rates](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-from-rr-intervals-to-heart-rates). The cell below solves the example with the constructs learnt so far: a list, a loop, a condition and a function.

In [ ]:
# The worked example of the lecture page: from RR intervals to heart rates
rr_ms = [820, 790, 805]                # three RR intervals of one patient (ms)
rates = []                             # an empty list for the heart rates
for rr in rr_ms:                       # the intervals, one after the other
    rate = round(60000 / rr, 1)        # one minute holds 60,000 ms
    rates.append(rate)
    if rate > 100:
        print(rr, "ms ->", rate, "bpm: tachycardia")
    else:
        print(rr, "ms ->", rate, "bpm")
print("heart rates (bpm):", rates)


def heart_rates(rr_list):
    """Heart rate (bpm) of every RR interval (ms) of a list."""
    out = []
    for rr in rr_list:
        out.append(round(60000 / rr, 1))
    return out


print("the same steps for another patient:", heart_rates([1000, 750, 600]))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-b1-strip)*

**Python note.** `rates = []` creates an empty list, and `rates.append(rate)` adds one value at its end. The loop `for rr in rr_ms:` runs its indented lines once for every interval, and inside it `if rate > 100:` chooses between the two messages. The function `heart_rates` puts the same steps under one name, so that one line applies them to the next patient: For 1000, 750 and 600 ms it gives 60, 80 and 100 bpm, the answer of the knowledge check on the lecture page.

In [ ]:
# Python practice: a small example for learning the language
rr = np.array([0.81, 0.85, 0.83])   # three RR intervals (s)
print(60 / rr)                      # heart rate of each beat (bpm)
print(rr.mean(), rr.std(ddof=1))

**Python note.** NumPy, imported in the start cell as `np`, works on whole arrays of numbers at once: `60 / rr` divides 60 by each RR interval. `.mean()` returns the mean, and `.std(ddof=1)` the sample standard deviation, which divides by $n-1$.

**Quick check.** After `troponin.append(4.0)`, what does `troponin[-1]` return?

<details><summary>Answer</summary><code>4.0</code>: the index <code>-1</code> always returns the last element, and <code>append()</code> put <code>4.0</code> at the end.</details>

**Exercise 1.1.** Compute the RR interval in seconds for a heart rate of 72 bpm and round it to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(60 / 72, 3)
teach.check("Exercise 1.1", answer)

**Exercise 1.2.** Write a function `qtc_fridericia(qt_ms, rr_s)` that divides the QT interval by the cube root of the RR interval, $QT_{cF} = QT / RR^{1/3}$. Use it for the case of the week (QT 391 ms, heart rate 85 bpm) and round to one decimal.

In [ ]:
def qtc_fridericia(qt_ms, rr_s):
    return None   # replace None with the formula


answer = qtc_fridericia(391, 60 / 85)
answer = round(answer, 1) if answer is not None else None
teach.check("Exercise 1.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def qtc_fridericia(qt_ms, rr_s):
    """QT interval corrected with Fridericia's formula (ms)."""
    return qt_ms / rr_s ** (1 / 3)


answer = round(qtc_fridericia(391, 60 / 85), 1)
teach.check("Exercise 1.2", answer)

**Exercise 1.3.** Count the patients of the list `clinic` whose heart rate is above 80 bpm. Use a `for` loop and an `if`.

In [ ]:
count = 0
# your loop here
answer = None   # your answer
teach.check("Exercise 1.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
count = 0
for p in clinic:
    if p["hr"] > 80:
        count = count + 1
answer = count
teach.check("Exercise 1.3", answer)

**A patient table with pandas.** pandas keeps a table in a *DataFrame*: Rows are patients and columns are variables. The small table below holds six patients, with the same columns as the Cleveland data of the mini project: Age in years, sex (1 = male, 0 = female), total cholesterol in mg/dl and coronary artery disease (1 = present). The same commands work on a table of 303 or of 300,000 rows.

In [ ]:
# Python practice: a small example for learning the language
import pandas as pd

patients = pd.DataFrame({
    "age": [63, 67, 37, 41, 56, 62],
    "sex": [1, 1, 1, 0, 1, 0],
    "chol": [233, 286, 250, 204, 236, 268],
    "disease": [0, 1, 0, 0, 0, 1],
})
print(patients)
print("rows and columns:", patients.shape)
print("mean cholesterol (mg/dl):", round(patients["chol"].mean(), 1))

**Python note.** `import pandas as pd` loads pandas under its usual short name `pd`. `pd.DataFrame({...})` builds a table from a dictionary: Each key is the name of a column, and the list after it holds the values of the column. `patients["chol"]` selects one column, and `.mean()` averages it. `.shape` gives the number of rows and columns.

In [ ]:
# Python practice: a small example for learning the language
older = patients[patients["age"] >= 60]            # the rows that meet a condition
print(older)
print("patients aged 60 or more:", len(older))
print(patients["sex"].value_counts())               # how often each value occurs
print(patients.groupby("disease")["chol"].mean())   # mean cholesterol in each group

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-b1-pandas-2)*

**Python note.** `patients["age"] >= 60` compares every age with 60 and gives a column of `True` and `False`; inside the square brackets it keeps only the rows with `True`. `value_counts()` counts every value of a column. `groupby("disease")` splits the table by the values of `disease`, and `["chol"].mean()` then averages the cholesterol within each group.

**On the lecture page:** [Worked example: five cholesterol values](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-five-cholesterol-values). The cell below computes the mean, the standard deviation and the median of the example, once with the formulas and once with pandas.

In [ ]:
# The worked example of the lecture page: five cholesterol values
import pandas as pd

chol5 = pd.Series([233, 286, 229, 250, 204])    # the first five patients of the Cleveland data (mg/dl)
mean = chol5.sum() / len(chol5)
sd = (((chol5 - mean) ** 2).sum() / (len(chol5) - 1)) ** 0.5
print(f"mean by hand: {mean:.2f}  pandas: {chol5.mean():.2f}")
print(f"SD by hand: {sd:.2f}  pandas: {chol5.std():.2f}")
print(f"median: {chol5.median():.1f}")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-b1-worked)*

**Python note.** `pd.Series` is a single column. The two ways give the same numbers: `.sum() / len(...)` is the formula of the mean, and `.mean()` the same calculation in pandas. `.std()` uses n − 1 in the denominator, the sample standard deviation of the lecture page.

In [ ]:
# Python practice: a small example for learning the language
colours = ["tab:red" if d == 1 else "tab:blue" for d in patients["disease"]]
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(range(1, 7), patients["chol"], color=colours)
ax.set_xlabel("patient")
ax.set_ylabel("total cholesterol (mg/dl)")
ax.set_title("Six patients (red: coronary artery disease)")
plt.show()

**Python note.** `plt.subplots` creates a figure (`fig`) and its axes (`ax`). `ax.bar` draws one bar per patient. The line `[... for d in ...]` is a *list comprehension*: It builds a list with one colour for every patient. Every axis names the quantity and its unit, and `plt.show()` shows the figure.

**Exercise 1.4.** In the table `patients`, compute the mean age of the patients with coronary artery disease (`disease == 1`) and round it to one decimal.

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.4", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(patients.loc[patients["disease"] == 1, "age"].mean(), 1)
teach.check("Exercise 1.4", answer)

**«Stop 1 · Python basics» is complete.** Next comes «Stop 1 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#last-two-hours-working-with-ecg-data) and continue reading at «Last two hours: Working with ECG data».

### Stop 1 · Prompt and application

The three prompts of this part summarise the Cleveland data and the virtual clinic with a mean, a percentage and a count.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Working with prompts.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab and opens with the Gemini button at the top of the window; ChatGPT, Claude, Copilot or Gemini in another browser tab work as well. All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md), where GitHub shows a copy button next to each one.

**Two names in every prompt: "data" and "answer".** An AI assistant usually does not see the notebook, so every prompt describes the data in words. For each prompt, the data cell puts its data into a table named "data", and the prompt describes the rows and the columns of this table. The prompt also asks the assistant to keep the result under the name "answer", and the check cell reads the result under this name. These are the only two names from the code that appear in the prompts of the course.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

**Start cell.** For «Stop 1 · Prompt and application», the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
from vclinic.datasets import load_cleveland
heart = load_cleveland(source=DATA_SOURCE)                   # the Cleveland data from the chosen source
clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)   # the virtual clinic

#### Prompt 1.1: The mean age of the patients with disease

**Goal.** Find out how old, on average, the Cleveland patients with coronary artery disease were.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know the mean age of the patients with coronary artery disease among 303 patients who underwent coronary angiography at the Cleveland Clinic.  
> Data: The notebook holds a table named "data". Each row is a patient: The column "age" gives the age in years. The column "CAD" says "yes" when at least one major coronary artery was narrowed by more than 50 percent of its diameter at angiography, and "no" otherwise.  
> Task: Select the patients with coronary artery disease and compute their mean age. Also print how many patients were selected.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean age in years with one decimal and keep it under the name "answer".  
> Test: For comparison, also print the mean age of all patients, which must be 54.4 years. The mean of the patients with disease must differ from it.  

In [ ]:
# Data of Prompt 1.1: the table "data" that the prompt describes
data = pd.DataFrame({"age": heart["age"],
                     "CAD": heart["disease"].map({1: "yes", 0: "no"})})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.1
teach.check_step("Prompt 1.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.1 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"age": heart["age"],
                     "CAD": heart["disease"].map({1: "yes", 0: "no"})})
patients = data[data["CAD"] == "yes"]
print("Patients with disease:", len(patients))
print("Mean age of all patients:", round(data["age"].mean(), 1))
answer = round(patients["age"].mean(), 1)
print("Mean age of the patients with disease:", answer)
teach.check_step("Prompt 1.1", "answer", globals())

**What the result shows.** The patients with disease are about two years older than the whole group. A single mean says little about a single patient; the tables of the lecture page show the spread as well.

#### Prompt 1.2: The share of women

**Goal.** Find out which percentage of the Cleveland patients were women.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know which percentage of the 303 patients who underwent coronary angiography at the Cleveland Clinic were women.  
> Data: The notebook holds a table named "data". Each row is a patient: The column "age" gives the age in years, and the column "sex" says "female" or "male".  
> Task: Divide the number of women by the number of all patients and express the share in percent. Compute the percentage of men in the same way.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the percentage of women with one decimal and keep it under the name "answer". Also print the percentage of men.  
> Test: The two percentages must add up to 100.  

In [ ]:
# Data of Prompt 1.2: the table "data" that the prompt describes
data = pd.DataFrame({"age": heart["age"],
                     "sex": heart["sex"].map({1: "male", 0: "female"})})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.2
teach.check_step("Prompt 1.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"age": heart["age"],
                     "sex": heart["sex"].map({1: "male", 0: "female"})})
women = 100 * (data["sex"] == "female").mean()
men = 100 * (data["sex"] == "male").mean()
answer = round(women, 1)
print("Women (%):", answer, " Men (%):", round(men, 1))
teach.check_step("Prompt 1.2", "answer", globals())

**What the result shows.** About two thirds of the Cleveland patients are men. A model trained on such data learns less about women, and Week 9 treats this as a question of bias.

#### Prompt 1.3: Patients with HFrEF in the virtual clinic

**Goal.** Count the patients of the virtual clinic who have heart failure with reduced ejection fraction, and see the ejection fraction of every diagnosis in one figure.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to count the patients with heart failure with reduced ejection fraction (HFrEF) among the 500 synthetic patients of the virtual clinic, and to see how the ejection fraction differs between the diagnoses.  
> Data: The notebook holds a table named "data". Each row is a patient: The column "patient" gives the number of the patient, the column "main diagnosis" the main diagnosis and the column "LVEF" the left ventricular ejection fraction in percent. The main diagnosis is one of "none", "CCS" (chronic coronary syndrome), "ACS" (acute coronary syndrome), "HFrEF" (heart failure with reduced ejection fraction) and "HFpEF" (heart failure with preserved ejection fraction).  
> Task: Count the patients whose main diagnosis is HFrEF. Then draw a box plot of the ejection fraction for each main diagnosis, with the diagnoses on the horizontal axis and the ejection fraction in percent on the vertical axis.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of patients with HFrEF and keep it under the name "answer". Show the plot as well.  
> Test: Also print the number of patients of every diagnosis; the five numbers must add up to 500. No patient with HFrEF may have an ejection fraction above 40 percent.  

In [ ]:
# Data of Prompt 1.3: the table "data" that the prompt describes
data = pd.DataFrame({"patient": clinic["patient_id"],
                     "main diagnosis": clinic["main_diagnosis"].replace({"none": "none"}),
                     "LVEF": clinic["lvef"]})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.3
teach.check_step("Prompt 1.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.3 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"patient": clinic["patient_id"],
                     "main diagnosis": clinic["main_diagnosis"].replace({"none": "none"}),
                     "LVEF": clinic["lvef"]})
print(data["main diagnosis"].value_counts())
answer = int((data["main diagnosis"] == "HFrEF").sum())
order = ["none", "CCS", "ACS", "HFpEF", "HFrEF"]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.boxplot([data.loc[data["main diagnosis"] == d, "LVEF"] for d in order])
ax.set_xticks(range(1, 6), order)
ax.set_xlabel("Main diagnosis")
ax.set_ylabel("LVEF (%)")
plt.show()
print("Patients with HFrEF:", answer)
teach.check_step("Prompt 1.3", "answer", globals())

**What the result shows.** In the registry, every patient with HFrEF has an ejection fraction of 40 percent or less, as the definition of the European Society of Cardiology requires, and every patient with HFpEF one of 50 percent or more. Heart failure with mildly reduced ejection fraction (41 to 49 percent) is left out to keep the virtual clinic simple.

**«Stop 1 · Prompt and application» is complete.** Next comes «Stop 1 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#last-two-hours-working-with-ecg-data) and continue reading at «Last two hours: Working with ECG data».

### Stop 1 · Ready application

This part holds the complete mini project: The Cleveland data are loaded from one of three sources, checked, summarised by disease status and drawn, and the virtual clinic opens beside them. The last cell computes the results of the three prompts.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 1 · Ready application», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

#### Mini project: The Kaggle Heart Disease data set: loading with pandas, statistical summary and visualisation

*Lecture page: [Mini project: Loading the Kaggle Heart Disease data set, statistical summary and visualisation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#mini-project-loading-the-kaggle-heart-disease-data-set-statistical-summary-and-visualisation)*

The Heart Disease data set shared on Kaggle comes from the Cleveland data of the UCI Heart Disease collection: 14 variables of 303 patients who underwent coronary angiography. The column `num` grades the disease from 0 (no narrowing of more than 50 percent) to 4. The course bundles the original file of the UCI repository, so no Kaggle account is needed. This section is the in-class mini project of the week: Load the data with pandas, process them, summarise them with statistics and visualise them with Matplotlib.

**From the formulas to the code.** The mean $\bar{x}$ is `df["chol"].mean()`. The sample standard deviation $s$, with $n-1$ in the denominator, is `df["chol"].std()`. The median is `.median()`, and the interquartile range is `.quantile(0.75) - .quantile(0.25)`.

In [ ]:
# Loading the Cleveland data from one of three sources
DATA_SOURCE = "web"   #@param ["library", "web", "computer"]

COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
FILE = "processed.cleveland.data"
URL = vclinic.COURSE_RAW + "data/uci-heart-disease/" + FILE   # the copy of the UCI file in the course repository

if DATA_SOURCE == "library":
    # ucimlrepo, the Python library of the UCI Machine Learning Repository, downloads the data set by its number
    vclinic.install("ucimlrepo")
    from ucimlrepo import fetch_ucirepo
    uci = fetch_ucirepo(id=45)
    heart = pd.concat([uci.data.features, uci.data.targets], axis=1)[COLUMNS]
elif DATA_SOURCE == "web":
    # pandas reads the file straight from its web address
    heart = pd.read_csv(URL, header=None, names=COLUMNS, na_values="?")
else:
    # the file on this computer: download it first with the link in the note below this cell
    path = vclinic.local_file(FILE)   # Jupyter looks in the folder of this notebook; Colab opens an upload window
    heart = pd.read_csv(path, header=None, names=COLUMNS, na_values="?")

heart["disease"] = (heart["num"] > 0).astype(int)   # 1 when a major vessel is narrowed by more than 50 percent
print("source:", DATA_SOURCE, "| rows and columns:", heart.shape)
heart.head()

**Python note.** The same table can come from three sources. `fetch_ucirepo(id=45)` asks the UCI repository for its data set number 45 and returns the variables and the outcome as two tables, which `pd.concat` puts side by side. `pd.read_csv` reads a file from a web address just as it reads one from the computer: `header=None` says that the file has no row of column names, `names=COLUMNS` gives the names, and `na_values="?"` turns the question marks of the file into missing values.

**Reading the data from your own computer.** For the source `"computer"`, first download [uci_heart_disease_cleveland.zip](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/raw/main/data/downloads/uci_heart_disease_cleveland.zip). Open the zip file and put the file `processed.cleveland.data` in the same folder as this Jupyter notebook. Then choose `"computer"` as the source and run the data cell again. In Google Colab, the cell shows a button for choosing the file on your computer. The zip file also holds the citation and the license of the data.

**Python note.** pandas, imported as `pd`, keeps a table in a *DataFrame*: rows are patients and columns are variables. The cell above reads the file and names the columns. `.shape` gives the number of rows and columns, and `.head()` shows the first five rows.

| Column | Meaning |
|---|---|
| `age`, `sex` | Age in years; sex (1 = male, 0 = female) |
| `cp` | Chest pain type (1 typical angina, 2 atypical angina, 3 non-anginal pain, 4 asymptomatic) |
| `trestbps` | Resting systolic blood pressure (mmHg) |
| `chol` | Serum cholesterol (mg/dL) |
| `fbs` | Fasting blood sugar above 120 mg/dL (1 = yes) |
| `restecg` | Resting ECG (0 normal, 1 ST-T abnormality, 2 left ventricular hypertrophy) |
| `thalach` | Maximum heart rate reached in the exercise test (bpm) |
| `exang` | Exercise-induced angina (1 = yes) |
| `oldpeak` | ST depression induced by exercise relative to rest (mm) |
| `slope` | Slope of the peak exercise ST segment (1 up, 2 flat, 3 down) |
| `ca` | Number of major vessels coloured by fluoroscopy (0 to 3) |
| `thal` | Thallium scan (3 normal, 6 fixed defect, 7 reversible defect) |
| `num`, `disease` | Angiographic disease grade 0 to 4; `disease` = 1 when `num` > 0 |

In [ ]:
# Summary of the main variables, by disease status
cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
print(heart["disease"].value_counts())
heart.groupby("disease")[cols].agg(["mean", "std"]).round(1)

**Python note.** `value_counts()` counts each value of a column. `groupby("disease")` splits the table into the patients with and without disease, and `.agg(["mean", "std"])` computes the two statistics for each group and each column.

In [ ]:
# Processing the table: missing values, a selection, a new column and a sorted view
print(heart.isna().sum()[heart.isna().sum() > 0])        # missing values per column
older = heart[heart["age"] >= 60]                          # rows that meet a condition
print("patients aged 60 or more:", len(older))
heart["chol_mmol"] = (heart["chol"] / 38.67).round(2)      # a new column: cholesterol in mmol/L
heart.sort_values("thalach")[["age", "sex", "thalach", "disease"]].head()

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c127)*

**Python note.** `heart.isna().sum()` counts the missing values of every column. `heart[heart["age"] >= 60]` keeps the rows for which the condition is true. A new column is created by assignment, here the total cholesterol converted from mg/dL to mmol/L by dividing by 38.67. `sort_values("thalach")` sorts the rows by the maximum heart rate, and a list of names in double brackets selects columns.

**A first cardiological plot with Matplotlib.** *Lecture page: [A first cardiological plot with Matplotlib](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#a-first-cardiological-plot-with-matplotlib)*

`plt.subplots(1, 2)` creates a figure with two sets of axes side by side. `hist` draws a histogram of the age in each group, and `boxplot` compares the maximum heart rate of the two groups. Every axis names its quantity and its unit.

In [ ]:
# Two first figures of the registry
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for value, color, label in [(0, "#1F5F8B", "no disease"),
                            (1, "#9B1B1B", "disease")]:
    sub = heart[heart["disease"] == value]
    axes[0].hist(sub["age"], bins=15, alpha=0.6, color=color, label=label)
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Number of patients")
axes[0].legend()
heart.boxplot(column="thalach", by="disease", ax=axes[1])
axes[1].set_title("")
axes[1].set_xlabel("Disease (0 no, 1 yes)")
axes[1].set_ylabel("Maximum heart rate (bpm)")
fig.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# The same comparison with Seaborn, which builds on Matplotlib
try:
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(5.5, 4))
    sns.boxplot(data=heart, x="disease", y="thalach", color="#C9A0A0", ax=ax)
    ax.set_xlabel("Disease (0 no, 1 yes)")
    ax.set_ylabel("Maximum heart rate (bpm)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Seaborn is not installed; the Matplotlib figure above shows the same comparison.")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c131)*

**Observe and reflect.** Which variable differs most between the two groups in the summary table? Is the direction of the difference clinically plausible? What does the small number of women tell about how far the data can be generalised?

**Advanced (optional).** Two columns hold missing values. Count them with `heart.isna().sum()`, then compare the disease rate by chest pain type with `pd.crosstab(heart["cp"], heart["disease"], normalize="index")`.

In [ ]:
# Advanced (optional)
print(heart.isna().sum()[heart.isna().sum() > 0])
pd.crosstab(heart["cp"], heart["disease"], normalize="index").round(2)

#### The virtual clinic opens

*Lecture page: [The virtual cardiology clinic, the project of the course](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-virtual-cardiology-clinic-the-project-of-the-course)*

`vclinic.make_cohort()` creates the registry of the virtual cardiology clinic. Each patient has demographic data, risk factors, laboratory values, a left ventricular ejection fraction (LVEF), a main diagnosis and the parameters of a resting 12-lead electrocardiogram (ECG). A patient keeps the same values in every week of the course. The data are synthetic: They follow broad clinical patterns but describe no real person.

In [ ]:
clinic = vclinic.make_cohort(n=500, seed=2026)
print(clinic.shape)
clinic.head()

**Python note.** `vclinic.make_cohort(n=500, seed=2026)` calls a function of the course package with two *keyword arguments*. The seed fixes the random numbers, so every run gives the same 500 patients.

In [ ]:
# Main diagnoses and rhythm of the clinic
print(clinic["main_diagnosis"].value_counts())
print(clinic["ecg_rhythm"].value_counts())
clinic[["age", "bmi", "sbp", "ldl", "hdl", "egfr", "lvef", "ntprobnp",
        "ecg_hr", "ecg_pr", "ecg_qrs", "ecg_qt"]].describe().round(1)

The diagnosis codes are the international abbreviations: `CCS` chronic coronary syndrome, `ACS` acute coronary syndrome, `HFrEF` heart failure with reduced ejection fraction, `HFpEF` heart failure with preserved ejection fraction and `none` no structural heart disease. `AF` in the rhythm column is atrial fibrillation. The full data dictionary is in DATASETS.md.

In [ ]:
# The virtual clinic next to the Cleveland data
compare = pd.DataFrame({
    "Cleveland (real)": [
        heart["age"].mean(), 100 * heart["sex"].mean(),
        heart["trestbps"].mean(), heart["chol"].mean(),
        100 * heart["disease"].mean()],
    "Virtual clinic (synthetic)": [
        clinic["age"].mean(), 100 * (clinic["sex"] == "M").mean(),
        clinic["sbp"].mean(), clinic["total_chol"].mean(),
        100 * clinic["cad"].mean()],
}, index=["Mean age (years)", "Men (%)",
          "Systolic pressure (mmHg)",
          "Total cholesterol (mg/dL)",
          "Coronary disease (%)"])
compare.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(heart["age"], bins=20, density=True, alpha=0.55, color="#1F5F8B",
        label="Cleveland (real)")
ax.hist(clinic["age"], bins=20, density=True, alpha=0.55, color="#9B1B1B",
        label="Virtual clinic (synthetic)")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

**Observe and reflect.** The two cohorts look alike in the table. In which ways could a synthetic registry still be wrong in a way that matters for a model trained on it? Think of rare conditions, of correlations between variables and of measurement noise.

**The results of the prompts.** The cell below computes the results of the prompts of Stop 1 with the code of this application and checks them as the check cells of the prompts do.

In [ ]:
# The results of the prompts of this stop, computed in Python
mean_age = round(heart.loc[heart["disease"] == 1, "age"].mean(), 1)
print("Mean age of the patients with coronary artery disease:", mean_age)
teach.check("Prompt 1.1", mean_age)
women = round(100 * (heart["sex"] == 0).mean(), 1)
print("Percentage of women:", women)
teach.check("Prompt 1.2", women)
hfref = int((clinic["main_diagnosis"] == "HFrEF").sum())
print("Patients with HFrEF in the virtual clinic:", hfref)
teach.check("Prompt 1.3", hfref)

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#last-two-hours-working-with-ecg-data) and continue reading at «Last two hours: Working with ECG data». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: The ECG as data

*Lecture page: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#stop-2)*

A digital ECG is a table of numbers: one column per lead, one row per sample. In this stop, lead II of the ECG of the case of the week, VC-0317, is drawn and measured, and the QT interval of another patient is corrected for the heart rate by two formulas.

**At the end of this stop:** A strip of lead II with 1250 samples in 2.5 seconds, a heart rate of 84.9 beats per minute from the R peaks, and a QTc of 403.9 ms by Fridericia's formula.

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 2 · Python basics

*Lecture page: [The ECG as data](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-ecg-as-data)*

An ECG lead is a list of voltages measured at regular intervals. The examples of this part use a simple wave in place of a real ECG; it is enough to see how Python handles a signal. The part covers time and samples, slices, conditions, a matrix, a loop and a figure.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 2 · Python basics», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

In [ ]:
# Python practice: a small example for learning the language
fs_demo = 500                             # samples per second (Hz)
t = np.arange(0, 2, 1 / fs_demo)          # two seconds, one time point every 2 ms
wave = np.sin(2 * np.pi * 1.25 * t)       # a smooth wave that repeats 1.25 times per second
print("number of samples:", len(t))
print("first five time points (s):", t[:5])
print("time step (ms):", 1000 / fs_demo)

**Python note.** `np.arange(0, 2, 1 / fs_demo)` creates the numbers from 0 to 2 in steps of 0.002; the end value 2 is not included. `np.sin(...)` works on the whole array at once and gives one value for every time point. `len(t)` counts the elements, and `t[:5]` takes the first five.

In [ ]:
# Python practice: a small example for learning the language
i = int(0.5 * fs_demo)                                  # the sample at 0.5 s
print("sample", i, "at", t[i], "s:", round(wave[i], 3))
window = wave[int(0.2 * fs_demo):int(0.6 * fs_demo)]    # the samples between 0.2 and 0.6 s
print("samples in the window:", len(window))

**Python note.** Python counts from 0, so the first sample is `wave[0]`. A time becomes the number of a sample when it is multiplied by the sampling frequency: 0.5 s × 500 Hz gives sample 250. A slice `a:b` takes the elements from `a` up to `b`, without `b` itself; the window of 0.4 s therefore holds 200 samples.

In [ ]:
# Python practice: a small example for learning the language
late = t >= 1.0                                         # True for every sample from 1 s on
print("samples from 1 s on:", late.sum())
top = np.argmax(wave[late])                             # the place of the largest value in that part
print("highest point after 1 s (s):", round(t[late][top], 3))

**Python note.** `t >= 1.0` compares every time with 1.0 and gives an array of `True` and `False`, a *mask*. `wave[late]` keeps only the values where the mask is `True`, and `late.sum()` counts them, because `True` counts as 1. `np.argmax` gives the position of the largest value.

In [ ]:
# Python practice: a small example for learning the language
directions = np.array([[1.0, 0.0, 0.0],                 # three directions in space, one per row
                       [0.0, 1.0, 0.0],
                       [0.5, 0.866, 0.0]])
v = np.array([0.6, 0.8, -0.2])                          # one vector (mV)
print(directions @ v)                                   # one number for every row
print("third row by hand:", round(0.5 * 0.6 + 0.866 * 0.8 + 0.0 * -0.2, 4))

**Python note.** `@` multiplies a matrix by a vector: Every row is multiplied element by element with the vector, and the products are added. The result is the projection of the vector on each direction. The lecture page builds the twelve leads of the ECG in the same way, from one heart vector and twelve lead directions.

In [ ]:
# Python practice: a small example for learning the language
values = [0.05, 0.3, 0.9, 1.4, 1.1, 0.7, 0.35, 0.1]     # a wave around its peak
k = values.index(max(values))                           # start at the peak
while k + 1 < len(values) and values[k + 1] > 0.3:
    k += 1                                              # step right while the value stays above 0.3
print("the wave stays above 0.3 up to position", k)

**Python note.** `while` repeats its indented lines as long as its condition is true. Here the loop starts at the peak and steps to the right until the next value falls to 0.3 or below. The measurement of the lecture page finds the end of the QRS complex and of the T wave with searches of this kind, one sample at a time. `k += 1` is short for `k = k + 1`.

In [ ]:
# Python practice: a small example for learning the language
def qtc_category(qtc_ms, sex):
    limit = 460 if sex == "F" else 450                  # upper limit of normal (ms); "F" is female
    if qtc_ms > 500:
        return "markedly prolonged"
    elif qtc_ms > limit:
        return "prolonged"
    else:
        return "normal"


for qtc in (430, 455, 470, 510):
    print(qtc, "ms:", qtc_category(qtc, "F"))

**Python note.** `elif` means *else if*: Python tests the conditions from top to bottom and runs only the first block whose condition is true. `limit = 460 if sex == "F" else 450` chooses one of two values in a single line. The limits are those of the 2009 recommendations for women and men.

In [ ]:
# Python practice: a small example for learning the language
def describe(values):
    """Mean and range of a list of numbers."""
    arr = np.asarray(values, dtype=float)
    return arr.mean(), arr.max() - arr.min()


mean_qrs, range_qrs = describe([92, 96, 88, 94, 90])    # QRS durations of five beats (ms)
print("mean:", mean_qrs, "ms | range:", range_qrs, "ms")

**Python note.** A function can give back several values at once: `return a, b` returns a pair, and `mean_qrs, range_qrs = ...` takes the pair apart into two names. The text in triple quotes under `def` is the *docstring* of the function, a short description of what it does.

In [ ]:
# Python practice: a small example for learning the language
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(t, wave, lw=1)
ax.axvline(1.0, color="tab:red", ls="--", lw=1)         # a vertical line at 1 s
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude")
ax.grid(alpha=0.3)
plt.show()

**Python note.** `ax.plot(x, y)` joins the points (x, y) with a line; for a signal, x is the time and y the value. `axvline` draws a vertical line, for example at a point of measurement, and `grid` adds light grid lines like those of ECG paper.

**Exercise 2.1.** On ECG paper one large box lasts 0.2 s. How many samples does one large box hold at 500 Hz?

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(0.2 * 500)
teach.check("Exercise 2.1", answer)

**Exercise 2.2.** The heart vector is $\mathbf{H} = (0.6, 0.8, -0.2)$ mV and the direction of lead aVL is $(0.75, -0.433, 0)$. Compute the voltage of lead aVL with `@` and round it to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(np.array([0.75, -0.433, 0.0]) @ np.array([0.6, 0.8, -0.2])), 3)
teach.check("Exercise 2.2", answer)

**«Stop 2 · Python basics» is complete.** Next comes «Stop 2 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records) and continue reading at «Access to the PhysioNet and MIT-BIH data sets».

### Stop 2 · Prompt and application

The three prompts of this part work with the ECG of the case of the week: a strip of lead II, the heart rate from the R peaks and the QTc by Fridericia's formula. The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 2 · Prompt and application», the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)   # the virtual clinic
case = vclinic.patient_ecg(clinic, "VC-0317")            # the ECG of the case of the week

#### Prompt 2.1: A strip of lead II

**Goal.** Draw the first 2.5 seconds of lead II and see how many numbers such a strip holds.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see the first 2.5 seconds of lead II from the 10-second resting ECG of a 67-year-old woman, and to know how many samples this strip holds.  
> Data: The notebook holds a table named "data". Each row is one moment of the recording: The column "time" gives the time in seconds, and the column "II" the voltage of lead II in millivolts. The recording was sampled 500 times per second.  
> Task: Keep the rows whose time is less than 2.5 seconds and count them. Draw this part with the time in seconds on the horizontal axis and the voltage in millivolts on the vertical axis.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of samples in these 2.5 seconds and keep it under the name "answer". Show the plot as well.  
> Test: The time axis of the plot must start at 0 and end at 2.5 seconds.  

In [ ]:
# Data of Prompt 2.1: the table "data" that the prompt describes
data = pd.DataFrame({"time": case.time, "II": case.lead("II")})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.1
teach.check_step("Prompt 2.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.1 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"time": case.time, "II": case.lead("II")})
strip = data[data["time"] < 2.5]
answer = len(strip)
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(strip["time"], strip["II"], color="k", lw=0.9)
ax.set_xlim(0, 2.5)
ax.set_xlabel("Time (s)")
ax.set_ylabel("mV")
plt.show()
print("Samples in 2.5 s:", answer)
teach.check_step("Prompt 2.1", "answer", globals())

**What the result shows.** At 500 samples per second, 2.5 seconds are 1250 numbers for each lead, and the whole 10-second recording of 12 leads holds 60,000 numbers. The strip on the screen is a picture of this table.

#### Prompt 2.2: The heart rate from the R peaks

**Goal.** Find the R peaks of lead II and compute the heart rate from the intervals between them.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to measure the heart rate of a 67-year-old woman from her 10-second resting ECG.  
> Data: The notebook holds a table named "data". Each row is one moment of the recording: The column "time" gives the time in seconds, and the column "II" the voltage of lead II in millivolts. The recording was sampled 500 times per second.  
> Task: Find the R peaks. Accept as R peaks the peaks that exceed half of the highest value and lie at least 0.3 seconds apart. Compute the heart rate in beats per minute from the mean time between successive R peaks.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the heart rate with one decimal and keep it under the name "answer". Draw lead II with the R peaks marked.  
> Test: About 14 R peaks must be found in the 10 seconds, and the heart rate must lie between 40 and 150 beats per minute.  

In [ ]:
# Data of Prompt 2.2: the table "data" that the prompt describes
data = pd.DataFrame({"time": case.time, "II": case.lead("II")})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.2
teach.check_step("Prompt 2.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"time": case.time, "II": case.lead("II")})
from scipy.signal import find_peaks
t = data["time"].to_numpy()
x = data["II"].to_numpy()
r, _ = find_peaks(x, height=x.max() / 2, distance=int(0.3 * 500))
answer = round(60 / np.diff(t[r]).mean(), 1)
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(t, x, color="k", lw=0.7)
ax.plot(t[r], x[r], "o", color="#B4312A")
ax.set_xlabel("Time (s)")
ax.set_ylabel("mV")
plt.show()
print("R peaks:", len(r), " heart rate (bpm):", answer)
teach.check_step("Prompt 2.2", "answer", globals())

**What the result shows.** The heart rate of the case is close to its value in the registry, 85 beats per minute. The rule of the prompt is simple: In a patient with tall T waves, it can count T waves as R peaks and double the heart rate. The lecture page shows this with the ECGs of other patients.

#### Prompt 2.3: The QTc by two formulas

**Goal.** Correct the QT interval of patient VC-0254 for his fast heart rate with the formulas of Fridericia and Bazett, and compare the two.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to correct the QT interval of a 58-year-old man with a fast heart rate by two formulas and to compare the results.  
> Data: The notebook holds a table named "data" with one row: The column "patient" gives the number of the patient, the column "heart rate" the heart rate in beats per minute and the column "QT" the QT interval in milliseconds.  
> Task: Compute the RR interval in seconds as 60 divided by the heart rate. Then correct the QT interval with two formulas: Fridericia's formula divides the QT by the cube root of RR, and Bazett's formula divides it by the square root of RR.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the QTc of Fridericia in milliseconds with one decimal and keep it under the name "answer". Also print the QTc of Bazett.  
> Test: At a heart rate of 60 beats per minute, both formulas must give the QT itself.  

In [ ]:
# Data of Prompt 2.3: the table "data" that the prompt describes
data = pd.DataFrame({"patient": ["VC-0254"], "heart rate": [98], "QT": [343]})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.3
teach.check_step("Prompt 2.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.3 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"patient": ["VC-0254"], "heart rate": [98], "QT": [343]})
rate = data["heart rate"].iloc[0]
qt = data["QT"].iloc[0]
rr = 60 / rate
answer = round(qt / rr ** (1 / 3), 1)
bazett = round(qt / rr ** 0.5, 1)
print("RR (s):", round(rr, 3), " QTc Fridericia:", answer, " QTc Bazett:", bazett)
teach.check_step("Prompt 2.3", "answer", globals())

**What the result shows.** At 98 beats per minute, Bazett's formula gives about 35 ms more than Fridericia's. Bazett's formula overcorrects at fast heart rates, which is why most studies of drug safety report Fridericia's correction.

**«Stop 2 · Prompt and application» is complete.** Next comes «Stop 2 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records) and continue reading at «Access to the PhysioNet and MIT-BIH data sets».

### Stop 2 · Ready application

This part treats the ECG of the case of the week as data: Samples and sampling frequency, twelve leads from one heart vector, and the measurement of the PR, QRS and QT intervals in five steps. The measurement is checked on 100 patients and against your own caliper measurements.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 2 · Ready application», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

#### The ECG as data: Samples, sampling frequency and units

*Lecture page: [The ECG as data](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-ecg-as-data)*

**From the formulas to the code.** The sampling frequency $f_s$ is `case.fs`. The time of sample $n$, $t_n = n / f_s$, is in `case.time`. The number of samples in an interval, $N = \Delta t \cdot f_s$, is `round(dt * case.fs)`. The voltages in millivolts are the columns of `case.leads`.

In [ ]:
clinic = vclinic.make_cohort(n=500, seed=2026)   # the virtual clinic, as in Stop 1
case = vclinic.patient_ecg(clinic, "VC-0317")   # the case of the week
print(case)
print("samples per lead:", len(case.leads))
print("time step (s):", 1 / case.fs)
case.leads.head()

In [ ]:
# Lead II on ECG paper: 10 seconds and one beat
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5))
teach.plot_strip(axes[0], case.time, case.lead("II"), label="II")
one = (case.time >= 1.0) & (case.time <= 2.0)
teach.plot_strip(axes[1], case.time[one], case.lead("II")[one], label="II")
axes[1].plot(case.time[one][::10], case.lead("II")[one][::10], "o",
             color="#9B1B1B", ms=3, label="every 10th sample")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

**Python note.** `case.time >= 1.0` compares every time value with 1.0 and returns an array of `True` and `False`, a *mask*. Two masks joined with `&` keep the samples between 1 and 2 seconds, and `x[mask]` selects them. `[::10]` takes every tenth element.

**On the lecture page:** [Worked example: the RR interval of the case in samples](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-the-rr-interval-of-the-case-in-samples). The cell below computes the example at 500 Hz and, for comparison, at 360 and 1000 Hz.

In [ ]:
# Samples in one RR interval at three sampling frequencies
rr = 60 / 85
for fs in [500, 360, 1000]:
    print(f"fs = {fs} Hz: RR = {rr:.3f} s spans {round(rr * fs)} samples")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c161)*

**Explore.** The panel samples one heartbeat of the case again at a lower frequency and stores it with fewer bits, as an analog-to-digital converter (ADC) would do. The reference is the same beat at 2000 Hz. Move the sliders and watch how the height and the time of the R peak change. Below about 100 Hz the R peak is clipped and shifted. With few bits the trace turns into steps.

In [ ]:
# Explore: sampling frequency and resolution
fine = vclinic.simulate(case.params, duration_s=2.0, fs=2000, noise="none", seed=7)
t_ref, x_ref = fine.time, fine.lead("II")


def show_sampling(fs_hz=500, bits=12):
    step = int(round(2000 / fs_hz))
    t_s, x_s = t_ref[::step], x_ref[::step]
    q = 10.0 / 2 ** bits                       # step size over a 10 mV range
    x_q = np.round(x_s / q) * q
    window = (t_ref > 0.25) & (t_ref < 0.75)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(t_ref[window], x_ref[window], color="#888888", lw=1,
            label="2000 Hz reference")
    w2 = (t_s > 0.25) & (t_s < 0.75)
    ax.step(t_s[w2], x_q[w2], where="mid", color="#9B1B1B", lw=1.2,
            label=f"{fs_hz} Hz, {bits} bit")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("mV")
    ax.legend(loc="upper right")
    plt.show()
    print(f"R peak: reference {x_ref.max():.3f} mV, "
          f"sampled {x_q.max():.3f} mV; "
          f"time shift {abs(t_s[np.argmax(x_q)] - t_ref[np.argmax(x_ref)]) * 1000:.1f} ms; "
          f"resolution {q * 1000:.2f} µV")


try:
    from ipywidgets import interact, IntSlider
    interact(show_sampling,
             fs_hz=IntSlider(value=500, min=50, max=1000, step=50, description="fs (Hz)"),
             bits=IntSlider(value=12, min=4, max=16, step=1, description="bits"))
except ImportError:
    show_sampling(500, 12)
    show_sampling(100, 6)

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c163)*

#### Twelve leads from one heart vector

*Lecture page: [The heart as a vector: Twelve leads, one source](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-heart-as-a-vector-twelve-leads-one-source)*

**From the formulas to the code.** The heart vector $\mathbf{H} = (H_x, H_y, H_z)$ is a NumPy array of three numbers. The lead vectors $\mathbf{L}$ of the 12 leads are the rows of `vclinic.LEAD_MATRIX`. The lead voltage $V_L = \mathbf{L} \cdot \mathbf{H}$ for all leads at once is `vclinic.project(H)`, which computes `LEAD_MATRIX @ H`. The frontal axis $\alpha = \operatorname{atan2}(H_y, H_x)$ is `vclinic.frontal_axis(V_I, V_aVF)`.

**Start cell.** For the section «Twelve leads from one heart vector», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Twelve leads from one heart vector»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# repeated from the earlier cells: the data and the code that the cells below use
clinic = vclinic.make_cohort(n=500, seed=2026)   # the virtual clinic, as in Stop 1
case = vclinic.patient_ecg(clinic, "VC-0317")   # the case of the week

In [ ]:
# The lead vectors of the virtual clinic: x to the left, y down, z to the front
lead_table = pd.DataFrame(vclinic.LEAD_MATRIX, index=vclinic.LEADS,
                          columns=["Lx", "Ly", "Lz"]).round(3)
lead_table

**On the lecture page:** [Worked example: one vector in six leads](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-one-vector-in-six-leads). The cell below computes the twelve projections of the vector of the example and checks the two laws.

In [ ]:
# The worked example of the lecture
H = np.array([0.6, 0.8, -0.2])           # heart vector (mV)
V = vclinic.project(H)
volts = pd.Series(V, index=vclinic.LEADS).round(3)
print(volts.to_string())
print("Einthoven: I + III =", round(volts["I"] + volts["III"], 3), "and II =", volts["II"])
print("Goldberger: aVR + aVL + aVF =", round(volts["aVR"] + volts["aVL"] + volts["aVF"], 3))
print("Frontal axis (degrees):", round(vclinic.frontal_axis(V[0], V[5]), 1))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c171)*

**Python note.** `@` multiplies a matrix by a vector. Each row of the 12 x 3 lead matrix is multiplied element by element with the three numbers of `H`, and the products are added. A `pd.Series` is a single labelled column, here indexed by the lead names.

In [ ]:
# The 12-lead ECG of the case
fig = teach.plot_12lead(case.time, case.leads, title="VC-0317")
plt.show()
L = case.leads
print("Largest deviation from Einthoven's law (mV):",
      float(np.abs(L["II"] - (L["I"] + L["III"])).max()))

**Explore.** Turn the heart vector in the frontal plane and toward the front or the back of the chest, and watch the twelve projections. A lead shows its largest positive deflection when the vector points along it and a negative one when the vector points away.

In [ ]:
# Explore: one vector, twelve projections
def show_projection(frontal=50, anterior=-15, size=1.3):
    h = size * vclinic.direction(frontal, anterior)
    v = vclinic.project(h)
    colors = ["#9B1B1B" if x >= 0 else "#1F5F8B" for x in v]
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.bar(vclinic.LEADS, v, color=colors)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylim(-2.5, 2.5)
    ax.set_ylabel("Projection (mV)")
    ax.set_title(f"H = ({h[0]:.2f}, {h[1]:.2f}, {h[2]:.2f}) mV, axis "
                 f"{vclinic.frontal_axis(v[0], v[5]):.0f}°")
    plt.show()


try:
    from ipywidgets import interact, IntSlider, FloatSlider
    interact(show_projection,
             frontal=IntSlider(value=50, min=-180, max=180, step=5, description="frontal (°)"),
             anterior=IntSlider(value=-15, min=-90, max=90, step=5, description="front/back (°)"),
             size=FloatSlider(value=1.3, min=0.2, max=2.0, step=0.1, description="size (mV)"))
except ImportError:
    show_projection(50, -15, 1.3)

**Observe and reflect.** The same vector gives a positive deflection in lead II and a negative one in lead V1. Explain the difference with the angle between the vector and each lead. Why do the chest leads show larger deflections than the limb leads for vectors of the same size?

#### Measuring PR, QRS and QT

*Lecture page: [Intervals: PR, QRS and QT](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#intervals-pr-qrs-and-qt)*

**From the formulas to the code.** The heart rate $HR = 60 / RR$ is `60 / rr_s`. Bazett's correction $QT_{cB} = QT / \sqrt{RR}$ is `qt / np.sqrt(rr_s)`, and Fridericia's $QT_{cF} = QT / \sqrt[3]{RR}$ is `qt / np.cbrt(rr_s)`; both need the RR interval in seconds. The end of the T wave by the tangent method, $t_{end} = t_s + (b - x(t_s)) / x'(t_s)$, is the line `t_s + (base - x[t_s]) / slope * fs` in the function `t_end_tangent`.

The section builds the measurement in five steps and then joins them in one function, `measure_ecg`. The same steps are kept in `vclinic.measure` for the later weeks.

**Start cell.** For the section «Measuring PR, QRS and QT», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Measuring PR, QRS and QT»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# repeated from the earlier cells: the data and the code that the cells below use
clinic = vclinic.make_cohort(n=500, seed=2026)   # the virtual clinic, as in Stop 1
case = vclinic.patient_ecg(clinic, "VC-0317")   # the case of the week

In [ ]:
# Step 1. R peaks: band-pass, derivative, squaring and a 150 ms moving window
from scipy.signal import butter, filtfilt, find_peaks


def bandpass(x, fs, low, high):
    b, a = butter(2, [low / (fs / 2), high / (fs / 2)], btype="band")
    return filtfilt(b, a, x)


def lowpass(x, fs, cutoff=20.0):
    b, a = butter(2, cutoff / (fs / 2), btype="low")
    return filtfilt(b, a, x)


def detect_r_peaks(x, fs):
    y = bandpass(x, fs, 5, 15)                 # keeps the QRS frequencies
    d = np.gradient(y) * fs                    # slope (mV/s)
    z = d ** 2                                 # only large slopes stand out
    w = int(round(0.150 * fs))                 # 150 ms window
    m = np.convolve(z, np.ones(w) / w, mode="same")
    peaks, _ = find_peaks(m, height=0.3 * np.percentile(m, 99),
                          distance=int(0.25 * fs))
    yy = bandpass(x, fs, 1, 40)
    half = int(0.075 * fs)
    r = []
    for p in peaks:                            # largest deflection within 75 ms
        lo, hi = max(0, p - half), min(len(x), p + half + 1)
        r.append(lo + int(np.argmax(np.abs(yy[lo:hi]))))
    return np.unique(r)


r_idx = detect_r_peaks(case.lead("II"), case.fs)
print("R peaks found:", len(r_idx))
print("times (s):", np.round(r_idx / case.fs, 3))
print("true R times (s):", np.round(case.truth["all_beats_s"], 3))

In [ ]:
# Step 2. QRS onset and offset from the slope of all 12 leads
def global_slope(leads, fs):
    arr = np.column_stack([bandpass(leads[:, j], fs, 1, 40)
                           for j in range(leads.shape[1])])
    g = np.abs(np.gradient(arr, axis=0)).sum(axis=1) * fs
    w = max(1, int(round(0.010 * fs)))
    return np.convolve(g, np.ones(w) / w, mode="same")


def qrs_bounds(leads, fs, r_idx, frac=0.08):
    g = global_slope(leads, fs)
    onsets, offsets = [], []
    for r in r_idx:
        lo, hi = max(0, r - int(0.12 * fs)), min(len(g) - 1, r + int(0.14 * fs))
        level = frac * g[lo:hi + 1].max()      # 8 percent of the largest slope
        on = r
        while on > lo and g[on] > level:
            on -= 1
        off = r
        while off < hi and g[off] > level:
            off += 1
        onsets.append(on)
        offsets.append(off)
    return np.array(onsets), np.array(offsets)


on_idx, off_idx = qrs_bounds(case.leads.to_numpy(), case.fs, r_idx)
qrs_ms = (off_idx - on_idx) / case.fs * 1000
print("QRS durations (ms):", np.round(qrs_ms, 0))

**Python note.** `while` repeats its indented lines as long as its condition holds. Here it walks backward from the R peak, one sample at a time, until the slope of the 12 leads falls below the level. `np.column_stack` places arrays side by side as the columns of a matrix.

In [ ]:
# Step 3. End of the T wave by the tangent method (lead II, low-passed at 20 Hz)
def t_end_tangent(x, fs, qrs_off, next_r, base):
    lo = qrs_off + int(0.08 * fs)
    hi = qrs_off + int(0.7 * (next_r - qrs_off))
    seg = x[lo:hi] - base
    k = int(np.argmax(np.abs(seg)))
    peak = lo + k                              # T peak
    sign = 1.0 if seg[k] >= 0 else -1.0
    d = np.gradient(x) * fs
    stop = min(len(x) - 1, peak + int(0.20 * fs))
    t_s = peak + int(np.argmin(sign * d[peak:stop]))   # steepest point
    slope = d[t_s]
    if slope == 0:
        return np.nan
    return t_s + (base - x[t_s]) / slope * fs  # where the tangent meets the baseline


x20 = lowpass(case.lead("II"), case.fs, 20)
fs = case.fs
i = 2                                          # third beat
base = np.median(x20[on_idx[i] - int(0.03 * fs):on_idx[i]])
t_end = t_end_tangent(x20, fs, off_idx[i], r_idx[i + 1], base)
print(f"QT of beat: {(t_end - on_idx[i]) / fs * 1000:.1f} ms, true: {case.truth['qt_ms']:.0f} ms")

**On the lecture page:** [Worked example: the end of the T wave on the third beat of the case](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-the-end-of-the-t-wave-on-the-third-beat-of-the-case). The cell below draws the tangent construction of the example on the third beat of the case.

In [ ]:
# The tangent construction on the third beat
d20 = np.gradient(x20) * fs
seg = x20[off_idx[i] + int(0.08 * fs):off_idx[i] + int(0.7 * (r_idx[i + 1] - off_idx[i]))]
peak = off_idx[i] + int(0.08 * fs) + int(np.argmax(np.abs(seg - base)))
t_s = peak + int(np.argmin(d20[peak:peak + int(0.2 * fs)]))
tt = np.linspace(t_s / fs - 0.05, t_end / fs + 0.02, 50)
fig, ax = plt.subplots(figsize=(9, 4))
w = slice(on_idx[i] - int(0.1 * fs), int(t_end) + int(0.12 * fs))
teach.plot_strip(ax, case.time[w], x20[w], label="II", v_range=(-0.5, 1.5))
ax.plot(tt, x20[t_s] + d20[t_s] * (tt - t_s / fs), color="#9B1B1B", lw=1.5,
        label="tangent at the steepest point")
ax.axhline(base, color="#1F5F8B", lw=1, ls="--", label="baseline")
ax.axvline(on_idx[i] / fs, color="k", lw=1, ls=":")
ax.axvline(t_end / fs, color="#9B1B1B", lw=1, ls=":")
ax.legend(loc="upper right", fontsize=8)
plt.show()

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c187)*

In [ ]:
# Step 4. Onset of the P wave
def p_onset(x, fs, qrs_on, earliest=0, frac=0.10):
    lo = max(0, int(earliest), qrs_on - int(0.30 * fs))
    hi = qrs_on - int(0.04 * fs)
    if hi - lo < int(0.08 * fs):               # too short to hold a P wave
        return np.nan
    seg = x[lo:hi]
    base = np.median(seg[:int(0.04 * fs)])     # flat segment before the P wave
    seg = seg - base
    k = int(np.argmax(seg))
    j = k
    while j > 0 and seg[j] > frac * seg[k]:    # back to 10 percent of the P height
        j -= 1
    return lo + j


po = p_onset(x20, fs, on_idx[i], earliest=t_end_tangent(
    x20, fs, off_idx[i - 1], r_idx[i], base) + 0.02 * fs)
print(f"PR of beat: {(on_idx[i] - po) / fs * 1000:.0f} ms, true: {case.truth['pr_ms']:.0f} ms")

In [ ]:
# Step 5. All steps in one function: the measurement of the ECG laboratory
def measure_ecg(leads, fs, lead="II", sinus=True):
    """Median heart rate, PR, QRS, QT and QTc of a 12-lead recording."""
    leads = np.asarray(leads, dtype=float)
    x = leads[:, ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3",
                  "V4", "V5", "V6"].index(lead)]
    r = detect_r_peaks(leads[:, 1], fs)
    on, off = qrs_bounds(leads, fs, r)
    x20 = lowpass(x, fs, 20)
    pr, qrs, qt = [], [], []
    t_prev = None
    for k in range(len(r) - 1):
        base = np.median(x20[max(0, on[k] - int(0.03 * fs)):on[k]])
        te = t_end_tangent(x20, fs, off[k], r[k + 1], base)
        qrs.append((off[k] - on[k]) / fs * 1000)
        qt.append((te - on[k]) / fs * 1000)
        if sinus and t_prev is not None and np.isfinite(t_prev):
            pr.append((on[k] - p_onset(x20, fs, on[k], t_prev + 0.02 * fs)) / fs * 1000)
        t_prev = te
    rr_s = np.mean(np.diff(r)) / fs
    out = {"hr_bpm": 60 / rr_s, "rr_s": rr_s,
           "pr_ms": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
           "qrs_ms": np.median(qrs), "qt_ms": np.nanmedian(qt)}
    out["qtc_bazett_ms"] = out["qt_ms"] / np.sqrt(rr_s)
    out["qtc_fridericia_ms"] = out["qt_ms"] / np.cbrt(rr_s)
    return out


def compare(ecg):
    m = measure_ecg(ecg.leads.to_numpy(), ecg.fs, sinus=ecg.truth["rhythm"] == "sinus")
    keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
    return pd.DataFrame({"measured": [m[k] for k in keys],
                         "true": [ecg.truth[k] for k in keys]},
                        index=keys).round(1)


compare(case)

**On the lecture page:** [Worked example: the case of the week](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-the-case-of-the-week). The cell below repeats the example with the same rounded steps.

In [ ]:
# The worked example of the lecture page: the QTc of the case
qt = 391                              # QT interval (ms)
rr = round(60 / 85, 3)                # RR interval at 85 bpm (s)
root2 = round(rr ** 0.5, 3)           # square root, rounded as on the lecture page
root3 = round(rr ** (1 / 3), 3)       # cube root
bazett = round(qt / root2, 1)
fridericia = round(qt / root3, 1)
print("RR =", rr, "s | square root", root2, "| cube root", root3)
print("QTc by Bazett:", bazett, "ms")
print("QTc by Fridericia:", fridericia, "ms")
print("difference:", round(bazett - fridericia, 1), "ms | threshold for women: 460 ms")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-w-qtc)*

**Observe and reflect.** For the case of the week, Bazett's formula gives a QTc above the threshold of 460 ms for women. Fridericia's formula gives a value well below it. At 85 bpm, which correction would you report, and why? The lecture section on QT correction gives the recommendation of the 2009 statement.

In [ ]:
# The five patients of the interactive lab, Part C
LAB_PATIENTS = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0018"]
rows = []
for pid in LAB_PATIENTS:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs, sinus=e.truth["rhythm"] == "sinus")
    rows.append({"patient": pid, "conduction": e.truth["conduction"],
                 "HR": e.truth["hr_bpm"], "PR_true": e.truth["pr_ms"], "PR_alg": m["pr_ms"],
                 "QRS_true": e.truth["qrs_ms"], "QRS_alg": m["qrs_ms"],
                 "QT_true": e.truth["qt_ms"], "QT_alg": m["qt_ms"]})
lab_table = pd.DataFrame(rows).set_index("patient").round(1)
lab_table

The last patient, VC-0018, has a left bundle branch block (LBBB). Its QRS is wide, and its T wave in lead II is small and opposite to the main QRS direction. The tangent method in lead II is therefore unreliable for this patient, and the cell *Advanced (optional)* below shows why.

In [ ]:
# Validation on 100 patients of the clinic (sinus rhythm, normal conduction)
sample = clinic[(clinic["ecg_rhythm"] == "sinus")
                & (clinic["ecg_conduction"] == "normal")].head(100)
errors, qt_pairs = [], []
for pid in sample["patient_id"]:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs)
    errors.append({k: m[k] - e.truth[k] for k in ["pr_ms", "qrs_ms", "qt_ms"]})
    qt_pairs.append((m["qt_ms"], e.truth["qt_ms"]))
errors = pd.DataFrame(errors)
qt_pairs = np.array(qt_pairs)
summary = pd.DataFrame({"bias (ms)": errors.mean(),
                        "SD (ms)": errors.std(),
                        "largest ": errors.abs().max()}).round(1)
summary

**On the lecture page:** [Worked example: calipers against the truth](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-calipers-against-the-truth). The cell below computes the bias and the limits of agreement of the five differences of the example.

In [ ]:
# The worked example of the lecture page: calipers against the true values
import numpy as np

d = np.array([12, -4, 8, 20, 4])      # caliper minus true QT in five measurements (ms)
bias = d.mean()
sd = d.std(ddof=1)                    # n - 1 in the denominator
print("bias:", round(bias, 1), "ms")
print("SD of the differences:", round(sd, 2), "ms")
print("limits of agreement:", round(bias - 1.96 * sd, 1), "to", round(bias + 1.96 * sd, 1), "ms")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-w-ba)*

**Your calipers against the algorithm and the truth.** The tab *Reflect and export* of the interactive lab exports your caliper measurements of Part C as `my_caliper_measurements.csv`. Save the file in the folder of this notebook and run the next cell. Where a download is not possible, the button *Copy caliper measurements* copies the same text: Paste it between the triple quotes of `PASTED_CALIPERS` in the next cell. Without your own measurements, the cell uses an example file of simulated manual measurements.

In [ ]:
# Bland-Altman analysis of the QT interval
import io
import os
from vclinic.datasets import data_path

# paste the copied caliper measurements between the triple quotes
PASTED_CALIPERS = """
"""

own = "my_caliper_measurements.csv"
if PASTED_CALIPERS.strip():
    source = "PASTED_CALIPERS"
    calipers = pd.read_csv(io.StringIO(PASTED_CALIPERS.strip()))
elif os.path.exists(own):
    source = own
    calipers = pd.read_csv(own)
else:
    source = data_path("examples", "example_caliper_measurements.csv")
    calipers = pd.read_csv(source)
print("Measurements from:", source)
qt_manual = calipers[calipers["measure"] == "QT"].set_index("patient_id")["value_ms"]
pts = [p for p in qt_manual.index if p != "VC-0018"]
truth_qt = [vclinic.patient_ecg(clinic, p).truth["qt_ms"] for p in pts]
alg_qt = [lab_table.loc[p, "QT_alg"] for p in pts]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
teach.plot_bland_altman(axes[0], qt_manual[pts].to_numpy(), truth_qt)
axes[0].set_title("Calipers minus truth")
teach.plot_bland_altman(axes[1], qt_pairs[:, 0], qt_pairs[:, 1])
axes[1].set_title("Algorithm minus truth, 100 patients")
plt.tight_layout()
plt.show()

**Python note.** `elif` means *else if*: Python tests the conditions from top to bottom and runs only the first block whose condition is true. `io.StringIO` makes a piece of text behave like a file, so `pd.read_csv` can read pasted text in the same way as a file on disk.

**Exercise 2.3.** Measure the PR interval of patient VC-0188 with `measure_ecg` and round it to the nearest millisecond. Does the result meet the definition of a first-degree atrioventricular block?

In [ ]:
# The table "data" for the prompt of this exercise (Part F of the interactive lab)
ecg_g = vclinic.patient_ecg(clinic, "VC-0188")
m_g = measure_ecg(ecg_g.leads.to_numpy(), ecg_g.fs)
data = pd.DataFrame({"heart rate": [round(m_g["hr_bpm"], 1)], "PR": [round(m_g["pr_ms"], 1)],
                     "QRS": [round(m_g["qrs_ms"], 1)], "QT": [round(m_g["qt_ms"], 1)]})
answer = None   # your answer
teach.check("Exercise 2.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e = vclinic.patient_ecg(clinic, "VC-0188")
answer = round(measure_ecg(e.leads.to_numpy(), e.fs)["pr_ms"])
teach.check("Exercise 2.3", answer)
print("First-degree block (PR > 200 ms):", answer > 200)

**Advanced (optional).** The 2009 recommendations advise measuring the QT in the lead with the longest QT, which is often V2 or V3. Extend `measure_ecg` so that it measures the QT in leads II, V2, V3 and V5 and reports the longest. Try it on VC-0018 and look at the T waves of each lead: Why does the tangent method struggle when the T wave is opposite to the QRS?

In [ ]:
# Advanced (optional): the QT in four leads
e = vclinic.patient_ecg(clinic, "VC-0018")
for lead in ["II", "V2", "V3", "V5"]:
    print(lead, round(measure_ecg(e.leads.to_numpy(), e.fs, lead=lead)["qt_ms"], 1))
print("true QT:", e.truth["qt_ms"])

**The results of the prompts.** The cell below computes the results of the prompts of Stop 2 with the code of this application and checks them as the check cells of the prompts do.

In [ ]:
# The results of the prompts of this stop, computed in Python
samples = int(np.sum(case.time < 2.5))
print("Samples of lead II in 2.5 s:", samples)
teach.check("Prompt 2.1", samples)
hr = round(60 / np.diff(r_idx / case.fs).mean(), 1)       # from the R peaks of Step 1 above
print("Heart rate (bpm):", hr)
teach.check("Prompt 2.2", hr)
qtc_254 = round(343 / (60 / 98) ** (1 / 3), 1)             # VC-0254: QT 343 ms at 98 bpm
print("QTc of VC-0254 by Fridericia (ms):", qtc_254)
teach.check("Prompt 2.3", qtc_254)

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records) and continue reading at «Access to the PhysioNet and MIT-BIH data sets». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: Real recordings and a careless prompt

*Lecture page: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#stop-3)*

Real recordings bring their own noise, annotations and units. In this stop, record 100 of the MIT-BIH Arrhythmia Database is read with the beat labels of its cardiologists, and the heart rate variability is computed from its normal beats. Then a real 12-lead ECG of the PTB database is drawn. Finally, a QTc calculation that an AI assistant wrote from a short and careless prompt is requested again with the prompt card, and its result is checked.

**At the end of this stop:** The number of atrial premature beats in record 100 (33), the SDNN of its normal-to-normal intervals (36.0 ms), the heart rate of the PTB record (81.8 beats per minute) and a corrected QTc of 466.5 ms.

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 3 · Python basics

*Lecture page: [Access to PhysioNet and MIT-BIH](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records)*

A real recording carries labels next to the signal: where each beat lies and what kind of beat it is. The small examples of this part teach the Python that such information needs. The part covers counting labels, intervals between beats, the statistics of heart rate variability, the attributes of a record, catching errors, default values and a grid of figures.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 3 · Python basics», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

In [ ]:
# Python practice: a small example for learning the language
labels = ["N", "N", "A", "N", "N", "V", "N", "A", "N", "N"]    # N normal, A atrial premature, V ventricular premature beat
print("number of beats:", len(labels))
print("atrial premature beats:", labels.count("A"))
print(pd.Series(labels).value_counts())

**Python note.** `labels.count("A")` counts how often one value occurs in a list. `pd.Series(labels)` turns the list into a pandas column, and `value_counts()` counts every value at once, from the most frequent to the rarest.

In [ ]:
# Python practice: a small example for learning the language
meaning = {"N": "normal", "A": "atrial premature", "V": "ventricular premature"}
for symbol in sorted(set(labels)):
    print(symbol, "=", meaning[symbol], ":", labels.count(symbol))

**Python note.** `set(labels)` keeps every value once, and `sorted` puts the values in order. The dictionary `meaning` translates each symbol into words, as the annotation guide of a database does.

In [ ]:
# Python practice: a small example for learning the language
fs_rec = 360                                            # sampling frequency of the recording (Hz)
r_samples = np.array([77, 370, 662, 946, 1231])         # sample numbers of five R peaks
rr_ms = np.diff(r_samples) / fs_rec * 1000              # intervals between neighbouring R peaks (ms)
print(rr_ms.round(1))
print("mean RR interval (ms):", round(rr_ms.mean(), 1))

**Python note.** `np.diff` subtracts every element from the next one, so five R peaks give four intervals. A number of samples divided by the sampling frequency gives seconds, and multiplied by 1000 it gives milliseconds.

In [ ]:
# Python practice: a small example for learning the language
rr = np.array([812, 805, 798, 820, 815, 790])           # six RR intervals (ms)
print("mean (ms):", round(rr.mean(), 1))
print("standard deviation (ms):", round(rr.std(ddof=1), 1))
diffs = np.diff(rr)
print("successive differences (ms):", diffs)
print("root mean square of the differences (ms):", round(np.sqrt(np.mean(diffs ** 2)), 1))

**Python note.** `rr.std(ddof=1)` is the sample standard deviation, with n − 1 in the denominator, as in the formula of SDNN. `diffs ** 2` squares every element, `np.mean` averages the squares and `np.sqrt` takes the square root: Together they give the RMSSD of the lecture page.

In [ ]:
# Python practice: a small example for learning the language
from types import SimpleNamespace

record_demo = SimpleNamespace(record_name="100", fs=360, sig_name=["MLII", "V5"], sig_len=650000)
print(record_demo.record_name, record_demo.sig_name)
print("duration (min):", round(record_demo.sig_len / record_demo.fs / 60, 1))

**Python note.** Some libraries return an *object* that keeps its data in named attributes, read with a dot: `record_demo.fs` is the sampling frequency. wfdb returns its records in this way, with the attributes `fs`, `sig_name`, `sig_len` and `p_signal`. `SimpleNamespace` builds such an object by hand, here for practice.

In [ ]:
# Python practice: a small example for learning the language
text = "12,5"                                           # a number written with a decimal comma
try:
    value = float(text)
except ValueError:
    value = float(text.replace(",", "."))               # Python expects a decimal point
print(value)

**Python note.** `try` runs a block, and `except` catches an error of the named kind instead of stopping the cell. `float("12,5")` fails because Python writes decimals with a point; the `except` block repairs the text. Reading a record from the internet uses the same pattern: Without a connection, `except` can turn to a copy on the computer.

In [ ]:
# Python practice: a small example for learning the language
def samples_to_ms(n_samples, fs=360):
    return n_samples / fs * 1000


print(samples_to_ms(180))               # uses the default fs = 360
print(samples_to_ms(180, fs=500))       # a recording at 500 Hz

**Python note.** `fs=360` is a *default value*: The function uses it when a call gives no `fs`. A default is convenient, and it is also the silent error of the code safety checklist: 180 samples are 500 ms at 360 Hz but 360 ms at 500 Hz.

In [ ]:
# Python practice: a small example for learning the language
t3 = np.arange(0, 1, 1 / 250)
fig, axes = plt.subplots(2, 3, figsize=(8, 3.6), sharex=True, sharey=True)
for k, ax in enumerate(axes.flat):
    ax.plot(t3, np.sin(2 * np.pi * (k + 1) * t3), lw=0.8)
    ax.set_title(f"wave {k + 1}", fontsize=9)
fig.supxlabel("time (s)")
plt.tight_layout()
plt.show()

**Python note.** `plt.subplots(2, 3)` creates a grid of axes with two rows and three columns. `axes.flat` goes through the axes one by one, and `enumerate` gives the number of each. The standard print of a 12-lead ECG is a grid of this kind, with three rows and four columns.

**Exercise 3.1.** Compute the RMSSD of the RR intervals 1000, 960, 1010, 990 and 1030 ms, and round it to two decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rr_ex = np.array([1000, 960, 1010, 990, 1030])
answer = round(float(np.sqrt(np.mean(np.diff(rr_ex) ** 2))), 2)
teach.check("Exercise 3.1", answer)

**Exercise 3.2.** An ECG was sampled 250 times per second, and two neighbouring R peaks lie at the samples 410 and 612. Compute the RR interval in milliseconds and round it to one decimal.

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round((612 - 410) / 250 * 1000, 1)
teach.check("Exercise 3.2", answer)

**«Stop 3 · Python basics» is complete.** Next comes «Stop 3 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-paper-of-the-week) and continue reading at «The paper of the week».

### Stop 3 · Prompt and application

The four prompts of this part work with real recordings of PhysioNet, and the last one turns a careless prompt into a prompt card. The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 3 · Prompt and application», the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
from vclinic.datasets import read_physionet, read_physionet_annotation
mit, fs_mit, notes = read_physionet("100", "mitdb", source=DATA_SOURCE)                    # MIT-BIH record 100
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=DATA_SOURCE)   # its beat labels
ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=DATA_SOURCE)   # a 12-lead ECG of the PTB database

#### Prompt 3.1: The beats of record 100

**Goal.** Count the beats of a 30-minute recording by the labels that cardiologists gave them.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Record 100 of the MIT-BIH Arrhythmia Database is a 30-minute ambulatory ECG of a 69-year-old man. The cardiologist wants to count the beats of this recording by the labels that cardiologists gave them.  
> Data: The notebook holds a table named "data". Each row is a mark that the cardiologists placed on the recording: The column "time" gives its place in seconds, and the column "label" its kind. "N" marks a normal beat, "A" an atrial premature beat and "V" a premature ventricular contraction; "+" marks a change of rhythm and is not a beat.  
> Task: Count how often each label occurs and print the counts.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of atrial premature beats and keep it under the name "answer".  
> Test: The labels N, A and V together must count 2273 beats.  

In [ ]:
# Data of Prompt 3.1: the table "data" that the prompt describes
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.1
teach.check_step("Prompt 3.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.1 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
counts = data["label"].value_counts()
print(counts)
answer = int(counts.get("A", 0))
print("Atrial premature beats:", answer)
teach.check_step("Prompt 3.1", "answer", globals())

**What the result shows.** Record 100 is a quiet recording of a 69-year-old man: most beats are normal, with a few dozen atrial premature beats and a single premature ventricular contraction. The labels turn it into a reference against which a detector can be measured.

#### Prompt 3.2: Heart rate variability: SDNN

**Goal.** Measure how much the intervals between normal beats vary over 30 minutes.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to measure the heart rate variability of record 100 of the MIT-BIH Arrhythmia Database, a 30-minute ambulatory ECG of a 69-year-old man, by the SDNN.  
> Data: The notebook holds a table named "data". Each row is a mark that the cardiologists placed on the recording: The column "time" gives its place in seconds, and the column "label" its kind. "N" marks a normal beat, "A" an atrial premature beat and "V" a premature ventricular contraction; "+" marks a change of rhythm and is not a beat.  
> Task: Keep only the beats, that is, all labels except "+". Compute the RR intervals between successive beats in milliseconds. Keep only the intervals between two normal beats, both labelled "N": These are the NN intervals. The SDNN is their standard deviation; compute it as for a sample, with the number of intervals minus one in the denominator. Also print the mean NN interval.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the SDNN in milliseconds with one decimal and keep it under the name "answer".  
> Test: The mean NN interval must lie between 700 and 900 ms.  

In [ ]:
# Data of Prompt 3.2: the table "data" that the prompt describes
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.2
teach.check_step("Prompt 3.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.2 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
beats = data[data["label"] != "+"]
labels = beats["label"].to_numpy()
rr_ms = np.diff(beats["time"].to_numpy()) * 1000
nn_ms = rr_ms[(labels[:-1] == "N") & (labels[1:] == "N")]
answer = round(np.std(nn_ms, ddof=1), 1)
print("NN intervals:", len(nn_ms), " mean (ms):", round(nn_ms.mean(), 1), " SDNN (ms):", answer)
teach.check_step("Prompt 3.2", "answer", globals())

**What the result shows.** The SDNN depends on the length of the recording, so a value of 30 minutes cannot be compared with one of 5 minutes or 24 hours. The standards of 1996 give separate reference values for recordings of 5 minutes and of 24 hours.

#### Prompt 3.3: A real 12-lead ECG

**Goal.** Draw a real 12-lead ECG of a patient with an acute myocardial infarction as on ECG paper, and measure its heart rate.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see the real 12-lead ECG of an 81-year-old woman admitted with an acute infero-lateral myocardial infarction in the layout of ECG paper, and to measure her heart rate.  
> Data: The notebook holds a table named "data". Each row is one moment of the recording: The column "time" gives the time in seconds, and the columns "I", "II", "III", "aVR", "aVL", "aVF" and "V1" to "V6" give the voltage of the twelve standard leads in millivolts. The recording lasts 38.4 seconds and was sampled 1000 times per second.  
> Task: Draw the first 3 seconds of the twelve leads in three rows and four columns, as on ECG paper: I, II and III in the first column, aVR, aVL and aVF in the second, V1 to V3 in the third and V4 to V6 in the fourth, with the name of the lead above each panel. Then find the R peaks of lead I. Accept as R peaks the peaks that exceed half of the highest value and lie at least 0.3 seconds apart. Compute the heart rate in beats per minute from the mean time between successive R peaks.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the heart rate with one decimal and keep it under the name "answer". Show the figure as well.  
> Test: The heart rate must lie between 40 and 150 beats per minute, and the number of R peaks must fit it: At 80 beats per minute, 38.4 seconds hold about 51 beats.  

In [ ]:
# Data of Prompt 3.3: the table "data" that the prompt describes
leads = {"i": "I", "ii": "II", "iii": "III", "avr": "aVR", "avl": "aVL", "avf": "aVF",
               "v1": "V1", "v2": "V2", "v3": "V3", "v4": "V4", "v5": "V5", "v6": "V6"}
data = ptb[list(leads)].rename(columns=leads)
data.insert(0, "time", np.arange(len(ptb)) / fs_ptb)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.3
teach.check_step("Prompt 3.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.3 (run it only if the code of the assistant does not pass the check)
leads = {"i": "I", "ii": "II", "iii": "III", "avr": "aVR", "avl": "aVL", "avf": "aVF",
               "v1": "V1", "v2": "V2", "v3": "V3", "v4": "V4", "v5": "V5", "v6": "V6"}
data = ptb[list(leads)].rename(columns=leads)
data.insert(0, "time", np.arange(len(ptb)) / fs_ptb)
from scipy.signal import find_peaks
t = data["time"].to_numpy()
layout = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
first3 = t < 3
fig, axes = plt.subplots(3, 4, figsize=(12, 6), sharex=True)
for ax, name in zip(axes.T.ravel(), layout):
    ax.plot(t[first3], data[name].to_numpy()[first3], color="k", lw=0.7)
    ax.set_title(name, fontsize=9)
plt.tight_layout()
plt.show()
x = data["I"].to_numpy()
r, _ = find_peaks(x, height=x.max() / 2, distance=int(0.3 * 1000))
answer = round(60 / np.diff(t[r]).mean(), 1)
print("R peaks:", len(r), " heart rate (bpm):", answer)
teach.check_step("Prompt 3.3", "answer", globals())

**What the result shows.** In the figure, look at the ST segments of the inferior leads II, III and aVF and of the lateral leads. An algorithm measures intervals; deciding on the pattern of an infarction is the work of the physician. The R peaks were counted in lead I, because in lead II of this recording the same simple rule misses more than a third of the beats: A rule must be checked in the lead to which it is applied.

#### Prompt 3.4: From a careless prompt to the prompt card

**Goal.** From the short prompt "write a python function that gives the qtc from qt and rr in samples", an AI assistant wrote code that returned an impossible QTc of 17 ms. The data cell of this prompt shows this result. With this prompt, the same calculation is requested again with the seven fields of the prompt card, and the result is checked.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: On an ECG sampled 500 times per second, the QT interval and the RR interval were measured as numbers of samples. The cardiologist wants the QT interval corrected by Bazett's method, in milliseconds.  
> Data: The notebook holds a table named "data" with one row: The column "QT" gives the QT interval and the column "RR" the RR interval, both as numbers of samples. The recording was sampled 500 times per second.  
> Task: Turn the QT interval into milliseconds and the RR interval into seconds. Correct the QT by Bazett's method: Divide the QT in milliseconds by the square root of the RR interval in seconds.  
> Constraints: Use only this table. Do not assume the number of samples per second; use the 500 given here. The code must not read files, download anything from the internet or install programs.  
> Output: Print the corrected QT in milliseconds with one decimal and keep it under the name "answer".  
> Test: The same calculation must give exactly 400 ms for a QT of 200 samples and an RR interval of 500 samples, because the RR interval is then exactly 1 second.  

In [ ]:
# Data of Prompt 3.4: the table "data" that the prompt describes
data = pd.DataFrame({"QT": [196], "RR": [353]})
answer = None   # the answer of an earlier prompt is cleared

def qtc_of_the_short_prompt(qt_samples, rr_samples, fs=360):
    qt_ms = qt_samples / fs * 1000
    rr_ms = rr_samples / fs * 1000
    return qt_ms / rr_ms ** 0.5


print("The code of the short prompt gives a QTc of", round(qtc_of_the_short_prompt(196, 353), 1), "ms")
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.4
teach.check_step("Prompt 3.4", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.4 (run it only if the code of the assistant does not pass the check)
data = pd.DataFrame({"QT": [196], "RR": [353]})
fs = 500
qt_ms = data["QT"].iloc[0] / fs * 1000      # QT in milliseconds
rr_s = data["RR"].iloc[0] / fs              # RR interval in seconds
assert abs((200 / fs * 1000) / (500 / fs) ** 0.5 - 400) < 1e-9   # test with a known answer
answer = round(qt_ms / rr_s ** 0.5, 1)
print("Corrected QT (ms):", answer)
teach.check_step("Prompt 3.4", "answer", globals())

**What the result shows.** Two silent errors made the first result impossible: The assistant assumed 360 samples per second, and it put the RR interval in milliseconds under the square root. The data field with its units, the constraint against an assumed sampling rate and the test with a known answer prevent both errors before the code is used for a patient.

**«Stop 3 · Prompt and application» is complete.** Next comes «Stop 3 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-paper-of-the-week) and continue reading at «The paper of the week».

### Stop 3 · Ready application

This part works with real recordings: MIT-BIH record 100 is read from one of three sources and processed with wfdb, its beats are found by three detectors, heart rate variability is computed, a real 12-lead ECG of the PTB database is drawn in the standard layout, and the code safety checklist is applied to code from an AI assistant.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 3 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 2 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# functions of earlier parts, kept unchanged in the course package
from vclinic.week01 import bandpass, detect_r_peaks, measure_ecg

# repeated from the earlier cells: the data and the code that the cells below use
clinic = vclinic.make_cohort(n=500, seed=2026)   # the virtual clinic, as in Stop 1
case = vclinic.patient_ecg(clinic, "VC-0317")   # the case of the week

#### Access to PhysioNet and MIT-BIH, reading and processing with wfdb, and R-peak detection with NeuroKit2: Record 100

*Lecture page: [Access to the PhysioNet and MIT-BIH data sets](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#real-data-physionet-records) · [Reading and processing ECGs with the wfdb library](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#reading-and-processing-ecgs-with-the-wfdb-library) · [R-peak detection (NeuroKit2)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#finding-every-beat-r-peak-detection)*

**From the formulas to the code.** The width of the integration window, $N = 0.150 \cdot f_s$, is `int(round(0.150 * fs))`, which is 54 samples at 360 Hz. Sensitivity $Se = TP / (TP + FN)$ and positive predictive value $PPV = TP / (TP + FP)$ are computed from the counts that `vclinic.measure.match_beats` returns. A detected beat counts as true when it lies within 150 ms of a reference beat.

Record 100 of the Massachusetts Institute of Technology and Beth Israel Hospital (MIT-BIH) Arrhythmia Database lasts 30 minutes and holds two channels at 360 Hz. Cardiologists annotated every beat. The course keeps a copy of the record in its GitHub repository, and the data cell reads the record from PhysioNet, from this copy or from files on your own computer.

In [ ]:
# Reading MIT-BIH record 100 from one of three sources
DATA_SOURCE = "web"   #@param ["library", "web", "computer"]

import urllib.request

import wfdb

RECORD = "100"
FILES = [RECORD + ".hea", RECORD + ".dat", RECORD + ".atr"]   # header, signals and beat labels
URL = vclinic.COURSE_RAW + "data/physionet/mitdb/"   # the copy of the PhysioNet files in the course repository

if DATA_SOURCE == "library":
    # wfdb reads the record straight from PhysioNet; pn_dir is the folder of the database
    record = wfdb.rdrecord(RECORD, pn_dir="mitdb")
    annotation = wfdb.rdann(RECORD, "atr", pn_dir="mitdb")
else:
    if DATA_SOURCE == "web":
        # the files are downloaded from their web address into the folder of this notebook
        for name in FILES:
            urllib.request.urlretrieve(URL + name, name)
    else:
        # the files on this computer: download them first with the link in the note below this cell
        vclinic.local_files(*FILES)   # Jupyter looks in the folder of this notebook; Colab opens an upload window
    record = wfdb.rdrecord(RECORD)
    annotation = wfdb.rdann(RECORD, "atr")

from vclinic.measure import match_beats

fs_mit = float(record.fs)
mit = pd.DataFrame(record.p_signal, columns=record.sig_name)
mit.index = pd.Index(np.round(np.arange(len(mit)) / fs_mit, 6), name="time_s")
ann_samples, ann_symbols = np.asarray(annotation.sample), list(annotation.symbol)
print("source:", DATA_SOURCE, "| fs:", fs_mit, "| signals:", record.sig_name, "| labels:", len(ann_samples))
print(pd.Series(ann_symbols).value_counts())
mit.head()

**Python note.** A PhysioNet record is a set of files with one name: The header `100.hea` describes the signals, `100.dat` holds them, and `100.atr` holds the beat labels. `wfdb.rdrecord` reads the header and the signals into one object, and `wfdb.rdann` the labels; with `pn_dir`, both read straight from PhysioNet. The last lines turn the object into the table `mit`, with the time in seconds as its index.

**Reading the data from your own computer.** For the source `"computer"`, first download [mitdb_record_100.zip](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/raw/main/data/downloads/mitdb_record_100.zip). Open the zip file and put the files `100.hea`, `100.dat` and `100.atr` in the same folder as this Jupyter notebook. Then choose `"computer"` as the source and run the data cell again. In Google Colab, the cell shows a button for choosing the files on your computer; select them together. The zip file also holds the citation and the license of the data.

The annotation symbols follow the convention of PhysioNet: `N` is a normal beat, `A` an atrial premature beat and `V` a premature ventricular contraction. `+` is not a beat; it marks a change of rhythm. The `wfdb` package reads the record when it is installed; otherwise the small reader of `vclinic.physionet` does the same work.

**Reading with wfdb.** `wfdb.rdrecord` reads the header and the signal file of a record, and `wfdb.rdann` reads its annotation file. The record keeps the sampling frequency in `fs`, the names of the signals in `sig_name`, their units in `units` and the signals in millivolts in `p_signal`. With `sampto`, only the first samples are read, here the first ten seconds, and `wfdb.plot_wfdb` draws them with the symbols of the annotations.

In [ ]:
# Record 100 with wfdb: its header and its first ten seconds
print("fs:", record.fs, "| signals:", record.sig_name, "| units:", record.units)
print("samples per signal:", record.sig_len, "| header comments:", record.comments)
print("labels:", len(annotation.sample), "| first symbols:", annotation.symbol[:8])
n10 = int(10 * record.fs)
if DATA_SOURCE == "library":
    first10 = wfdb.rdrecord(RECORD, pn_dir="mitdb", sampto=n10)
    ann10 = wfdb.rdann(RECORD, "atr", pn_dir="mitdb", sampto=n10)
else:
    first10 = wfdb.rdrecord(RECORD, sampto=n10)
    ann10 = wfdb.rdann(RECORD, "atr", sampto=n10)
fig = wfdb.plot_wfdb(record=first10, annotation=ann10, plot_sym=True, time_units="seconds",
                     title="MIT-BIH 100", figsize=(11, 5), return_fig=True)
for ax, name in zip(fig.axes, first10.sig_name):
    ax.set_ylabel(name + " (mV)")
fig.axes[-1].set_xlabel("Time (s)")
plt.show()

**Processing with wfdb.** `processing.resample_singlechan` resamples one signal to a new frequency and moves the annotations to the new samples. `processing.xqrs_detect` finds the QRS complexes with XQRS, the detector of the package.

In [ ]:
# Processing with wfdb: resampling to 250 Hz and the XQRS detector
try:
    from wfdb import processing

    x360 = record.p_signal[:, 0]
    x250, ann250 = processing.resample_singlechan(x360, annotation, record.fs, 250)
    print("samples at 360 Hz:", len(x360), "-> at 250 Hz:", len(x250))
    print("first label at sample:", annotation.sample[0], "->", ann250.sample[0])
    xqrs_peaks = processing.xqrs_detect(x360, fs=record.fs, verbose=False)
    print("XQRS:", len(xqrs_peaks), "QRS complexes")
except (ImportError, NameError):
    xqrs_peaks = None
    print("wfdb is not installed; this step is skipped.")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c214)*

In [ ]:
# The four stages of the detector on the first 3 seconds
x = mit["MLII"].to_numpy()
seg = slice(0, int(3 * fs_mit))
y = bandpass(x, fs_mit, 5, 15)
z = (np.gradient(y) * fs_mit) ** 2
w = int(round(0.150 * fs_mit))
m = np.convolve(z, np.ones(w) / w, mode="same")
t = np.arange(len(x)) / fs_mit
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
names = ["raw lead MLII (mV)", "band-pass 5-15 Hz",
         "squared slope", f"moving window ({w} samples)"]
for ax, sig, name in zip(axes, [x, y, z, m], names):
    ax.plot(t[seg], sig[seg], color="#9B1B1B" if ax is axes[3] else "k", lw=0.9)
    ax.set_ylabel(name, fontsize=8)
axes[3].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

**On the lecture page:** [Worked example: sensitivity and positive predictive value](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-sensitivity-and-positive-predictive-value). The cell below computes the two measures of the example; the cell after it measures the detector of the course on record 100 in the same way.

In [ ]:
# The worked example of the lecture page: sensitivity and positive predictive value
labelled, reported, matched = 2273, 2271, 2268   # labelled beats, detections, detections that match a label
tp = matched
fn = labelled - matched                          # missed beats
fp = reported - matched                          # false detections
print("TP =", tp, "| FN =", fn, "| FP =", fp)
print("Se =", round(100 * tp / (tp + fn), 2), "%")
print("PPV =", round(100 * tp / (tp + fp), 2), "%")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-w-seppv)*

In [ ]:
# Detection on the whole recording and comparison with the annotations
x = mit["MLII"].to_numpy()
beat_symbols = set("NLRBAaJSVrFejnE/fQ?")
ref = np.array([s for s, sym in zip(ann_samples, ann_symbols) if sym in beat_symbols])
found = detect_r_peaks(x, fs_mit)
tp, fp, fn = match_beats(found / fs_mit, ref / fs_mit, window_s=0.150)
print(f"reference beats: {len(ref)}, detected: {len(found)}")
print(f"TP = {tp}, FP = {fp}, FN = {fn}")
print(f"Se = {100 * tp / (tp + fn):.2f} %, PPV = {100 * tp / (tp + fp):.2f} %")

**R-peak detection with NeuroKit2.** `nk.ecg_clean` cleans the signal, by default with a high-pass filter at 0.5 Hz and a filter for the power line, and `nk.ecg_peaks` finds the R peaks with its default method. The table compares three detectors on the same annotated beats: the detector of the course, XQRS of wfdb and NeuroKit2.

In [ ]:
# R-peak detection with NeuroKit2, and three detectors compared
rows = [("detector of the course", found)]
if globals().get("xqrs_peaks") is not None:
    rows.append(("XQRS (wfdb)", np.asarray(xqrs_peaks)))
try:
    import neurokit2 as nk

    cleaned = nk.ecg_clean(x, sampling_rate=fs_mit)
    _, info = nk.ecg_peaks(cleaned, sampling_rate=fs_mit)
    nk_peaks = np.asarray(info["ECG_R_Peaks"])
    rows.append(("NeuroKit2", nk_peaks))
except ImportError:
    nk_peaks = None
    print("neurokit2 is not installed; pip install neurokit2 installs it.")
table = []
for name, peaks in rows:
    tp, fp, fn = match_beats(peaks / fs_mit, ref / fs_mit, window_s=0.150)
    table.append({"detector": name, "detected": len(peaks), "TP": tp, "FP": fp, "FN": fn,
                  "Se (%)": round(100 * tp / (tp + fn), 2), "PPV (%)": round(100 * tp / (tp + fp), 2)})
pd.DataFrame(table)

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c218)*

**Observe and reflect.** The detector finds every beat of record 100. Why is this result not enough to trust it on every recording? Think of noise, of wide ventricular beats and of tall T waves. Which records of a test set would you choose to challenge it?

**Advanced (optional).** With an internet connection, `wfdb.rdrecord("105", pn_dir="mitdb")` and `wfdb.rdann("105", "atr", pn_dir="mitdb")` read record 105, which is known for its noise, straight from PhysioNet. Run the detector on it and compare the result with `neurokit2.ecg_peaks` if neurokit2 is installed.

In [ ]:
# Advanced (optional): record 105 from PhysioNet (needs internet and wfdb)
try:
    import wfdb
    rec = wfdb.rdrecord("105", pn_dir="mitdb", sampto=360 * 300)
    ann = wfdb.rdann("105", "atr", pn_dir="mitdb", sampto=360 * 300)
    ref105 = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in beat_symbols])
    found105 = detect_r_peaks(rec.p_signal[:, 0], rec.fs)
    print(match_beats(found105 / rec.fs, ref105 / rec.fs))
except Exception as err:
    print("Not available here:", type(err).__name__)

#### Heart rate variability (HRV) calculation

*Lecture page: [Heart rate variability (HRV) calculation](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#heart-rate-variability)*

**From the formulas to the code.** With the RR intervals in milliseconds in an array `rr_ms`, the standard deviation of the normal-to-normal intervals, $SDNN = \sqrt{\frac{1}{N-1}\sum (RR_i - \overline{RR})^2}$, is `np.std(rr_ms, ddof=1)`. The root mean square of successive differences, $RMSSD = \sqrt{\frac{1}{N-1}\sum (RR_{i+1} - RR_i)^2}$, is `np.sqrt(np.mean(np.diff(rr_ms) ** 2))`.

**Start cell.** For the section «Heart rate variability (HRV) calculation», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 2 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Heart rate variability (HRV) calculation»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# functions of earlier parts, kept unchanged in the course package
from vclinic.week01 import detect_r_peaks

# repeated from the earlier cells: the data and the code that the cells below use
clinic = vclinic.make_cohort(n=500, seed=2026)   # the virtual clinic, as in Stop 1
case = vclinic.patient_ecg(clinic, "VC-0317")   # the case of the week

# Reading MIT-BIH record 100 from one of three sources
DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
import wfdb
RECORD = "100"
FILES = [RECORD + ".hea", RECORD + ".dat", RECORD + ".atr"]   # header, signals and beat labels
URL = vclinic.COURSE_RAW + "data/physionet/mitdb/"   # the copy of the PhysioNet files in the course repository
if DATA_SOURCE == "library":
    # wfdb reads the record straight from PhysioNet; pn_dir is the folder of the database
    record = wfdb.rdrecord(RECORD, pn_dir="mitdb")
    annotation = wfdb.rdann(RECORD, "atr", pn_dir="mitdb")
else:
    if DATA_SOURCE == "web":
        # the files are downloaded from their web address into the folder of this notebook
        for name in FILES:
            urllib.request.urlretrieve(URL + name, name)
    else:
        # the files on this computer: download them first with the link in the note below this cell
        vclinic.local_files(*FILES)   # Jupyter looks in the folder of this notebook; Colab opens an upload window
    record = wfdb.rdrecord(RECORD)
    annotation = wfdb.rdann(RECORD, "atr")
fs_mit = float(record.fs)
mit = pd.DataFrame(record.p_signal, columns=record.sig_name)
mit.index = pd.Index(np.round(np.arange(len(mit)) / fs_mit, 6), name="time_s")
ann_samples, ann_symbols = np.asarray(annotation.sample), list(annotation.symbol)

**On the lecture page:** [Worked example: six RR intervals](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#worked-example-six-rr-intervals). The cell below computes SDNN and RMSSD of the six intervals of the example.

In [ ]:
# The worked example of the lecture: six RR intervals (ms)
import numpy as np

rr_ms = np.array([812, 845, 830, 798, 860, 841])
print("mean RR:", rr_ms.mean())
print("SDNN:", round(np.std(rr_ms, ddof=1), 2), "ms")
print("successive differences:", np.diff(rr_ms))
print("RMSSD:", round(np.sqrt(np.mean(np.diff(rr_ms) ** 2)), 2), "ms")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c228)*

In [ ]:
# Five minutes of lead II for the case, as the 1996 Task Force recommends for short-term HRV
from vclinic.ecg import lead_ii_long

t5, x5, truth5 = lead_ii_long(case.params, duration_s=300, fs=250, seed=17)
r5 = detect_r_peaks(x5, 250)
rr5 = np.diff(r5) / 250 * 1000                 # RR intervals in ms
sdnn = np.std(rr5, ddof=1)
rmssd = np.sqrt(np.mean(np.diff(rr5) ** 2))
print(f"beats: {len(r5)}, mean HR: {60000 / rr5.mean():.1f} bpm, SDNN: {sdnn:.1f} ms, RMSSD: {rmssd:.1f} ms")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(r5[1:] / 250, rr5, color="#9B1B1B", lw=0.8)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("RR (ms)")
axes[0].set_title("Tachogram")
axes[1].scatter(rr5[:-1], rr5[1:], s=6, color="#1F5F8B")
axes[1].set_xlabel("RR$_i$ (ms)")
axes[1].set_ylabel("RR$_{i+1}$ (ms)")
axes[1].set_title("Poincaré plot")
plt.tight_layout()
plt.show()

In [ ]:
# Record 100: normal-to-normal intervals of the first 5 minutes
pairs = [(s1, s2) for (s1, a), (s2, b) in zip(zip(ann_samples, ann_symbols),
                                               zip(ann_samples[1:], ann_symbols[1:]))
         if a == "N" and b == "N" and s2 <= 300 * fs_mit]
nn_ms = np.array([(s2 - s1) / fs_mit * 1000 for s1, s2 in pairs])
print(f"NN intervals: {len(nn_ms)}, mean HR: {60000 / nn_ms.mean():.1f} bpm")
print(f"SDNN: {np.std(nn_ms, ddof=1):.1f} ms, RMSSD: {np.sqrt(np.mean(np.diff(nn_ms) ** 2)):.1f} ms")

**HRV with NeuroKit2.** `nk.hrv_time` computes SDNN, RMSSD and many other measures from R peaks, or from RR intervals in milliseconds with their times. The table compares its result for all detected beats of the first five minutes with its result for the NN intervals alone and with the calculation by hand.

In [ ]:
# HRV with NeuroKit2: all detected beats against the NN intervals
try:
    import neurokit2 as nk

    peaks_nk = globals().get("nk_peaks")
    if peaks_nk is None:
        _, info = nk.ecg_peaks(nk.ecg_clean(mit["MLII"].to_numpy(), sampling_rate=fs_mit), sampling_rate=fs_mit)
        peaks_nk = np.asarray(info["ECG_R_Peaks"])
    all_beats = nk.hrv_time(peaks_nk[peaks_nk <= 300 * fs_mit], sampling_rate=fs_mit)
    nn_times = np.array([s2 / fs_mit for s1, s2 in pairs])
    nn_only = nk.hrv_time({"RRI": nn_ms, "RRI_Time": nn_times})
    summary = pd.DataFrame({
        "SDNN (ms)": [all_beats["HRV_SDNN"].iloc[0], nn_only["HRV_SDNN"].iloc[0], np.std(nn_ms, ddof=1)],
        "RMSSD (ms)": [all_beats["HRV_RMSSD"].iloc[0], nn_only["HRV_RMSSD"].iloc[0], np.sqrt(np.mean(np.diff(nn_ms) ** 2))]},
        index=["NeuroKit2, all detected beats",
               "NeuroKit2, NN intervals",
               "by hand, NN intervals"]).round(1)
    display(summary)
except ImportError:
    print("neurokit2 is not installed; pip install neurokit2 installs it.")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c232)*

**Observe and reflect.** The RMSSD of all detected beats is about twice the RMSSD of the NN intervals. Which beats of the first five minutes cause this difference, and why does the definition of short-term HRV use NN intervals only?

**Explore.** The generator of the virtual clinic has a variability parameter `hrv`: 1.0 is a typical adult at rest, and smaller values mimic older age or heart failure. Move it and watch SDNN and RMSSD of a 5-minute recording.

In [ ]:
# Explore: variability and the two measures
def show_hrv(hrv=1.0, heart_rate=70):
    params = dict(case.params, hrv=hrv, hr_bpm=heart_rate)
    _, xs, _ = lead_ii_long(params, duration_s=300, fs=250, seed=5)
    rr = np.diff(detect_r_peaks(xs, 250)) / 250 * 1000
    print(f"hrv = {hrv:.2f}, HR = {heart_rate} bpm -> SDNN = {np.std(rr, ddof=1):.1f} ms, "
          f"RMSSD = {np.sqrt(np.mean(np.diff(rr) ** 2)):.1f} ms")


try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(show_hrv, hrv=FloatSlider(value=1.0, min=0.2, max=1.6, step=0.1),
             heart_rate=IntSlider(value=70, min=50, max=110, step=5, description="HR"))
except ImportError:
    show_hrv(1.0, 70)
    show_hrv(0.4, 70)

**Observe and reflect.** RMSSD reacts mainly to fast, beat-to-beat changes, and SDNN also to slow ones. Why does the 1996 Task Force recommend recordings of 5 minutes for short-term measures, and why should values from recordings of different lengths not be compared?

#### Visualising the 12 leads: A real 12-lead ECG from the PTB Diagnostic database

*Lecture page: [Visualising the 12 leads](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#visualising-the-12-leads)*

Record `s0010_re` of patient 1 of the PTB Diagnostic ECG Database, from the Physikalisch-Technische Bundesanstalt (PTB) in Germany, holds the 12 standard leads and three Frank leads at 1000 Hz for 38.4 seconds. Its header describes the patient. The section draws the 12 leads twice: first one below the other, then in the standard printout of the clinic. The record also shows how the measurement steps of «Stop 2 · Ready application» behave on a real recording.

**Start cell.** For the section «Visualising the 12 leads: A real 12-lead ECG from the PTB Diagnostic database», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. The functions written in «Stop 2 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Visualising the 12 leads: A real 12-lead ECG from the PTB Diagnostic database»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# functions of earlier parts, kept unchanged in the course package
from vclinic.week01 import detect_r_peaks, measure_ecg

In [ ]:
# Reading record s0010_re of patient 1 of the PTB database from one of three sources
DATA_SOURCE = "web"   #@param ["library", "web", "computer"]

import urllib.request

import wfdb

PTB_RECORD = "s0010_re"
PTB_FILES = [PTB_RECORD + ".hea", PTB_RECORD + ".dat", PTB_RECORD + ".xyz"]   # header, the 12 leads and the 3 Frank leads
PTB_URL = vclinic.COURSE_RAW + "data/physionet/ptbdb/patient001/"   # the copy of the PhysioNet files in the course repository

if DATA_SOURCE == "library":
    # wfdb reads the record straight from PhysioNet; pn_dir is the folder of the database
    record_ptb = wfdb.rdrecord(PTB_RECORD, pn_dir="ptbdb/patient001")
else:
    if DATA_SOURCE == "web":
        # the files are downloaded from their web address into the folder of this notebook
        for name in PTB_FILES:
            urllib.request.urlretrieve(PTB_URL + name, name)
    else:
        # the files on this computer: download them first with the link in the note below this cell
        vclinic.local_files(*PTB_FILES)   # Jupyter looks in the folder of this notebook; Colab opens an upload window
    record_ptb = wfdb.rdrecord(PTB_RECORD)

fs_ptb = float(record_ptb.fs)
ptb = pd.DataFrame(record_ptb.p_signal, columns=record_ptb.sig_name)
ptb.index = pd.Index(np.round(np.arange(len(ptb)) / fs_ptb, 6), name="time_s")
header = list(record_ptb.comments)
print("source:", DATA_SOURCE, "|", ptb.shape, fs_ptb)
for line in header[:9]:
    print(line)

**Reading the data from your own computer.** For the source `"computer"`, first download [ptbdb_s0010_re.zip](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/raw/main/data/downloads/ptbdb_s0010_re.zip). Open the zip file and put the files `s0010_re.hea`, `s0010_re.dat` and `s0010_re.xyz` in the same folder as this Jupyter notebook. Then choose `"computer"` as the source and run the data cell again. In Google Colab, the cell shows a button for choosing the files on your computer; select them together. The zip file also holds the citation and the license of the data.

In [ ]:
# The twelve leads of the real record (column names are lower case)
names = {"I": "i", "II": "ii", "III": "iii", "aVR": "avr", "aVL": "avl", "aVF": "avf",
         "V1": "v1", "V2": "v2", "V3": "v3", "V4": "v4", "V5": "v5", "V6": "v6"}
fig = teach.plot_12lead(ptb.index.to_numpy(), ptb, title="PTB patient001, s0010_re",
                        v_range=(-1.5, 1.5), names=names)
plt.show()

**The standard printout.** A 12-lead ECG is printed in three rows of four columns: I, II and III in the first column, aVR, aVL and aVF in the second, V1 to V3 in the third and V4 to V6 in the fourth. Each lead fills 2.5 seconds, and a fourth row shows lead II for all 10 seconds as a rhythm strip. On the paper, 25 mm/s and 10 mm/mV make a small square 0.04 s wide and 0.1 mV high, and a large square of five small squares 0.2 s wide and 0.5 mV high.

**Python note.** `plt.subplots(4, 1)` creates four rows of axes. `(t >= 2.5 * col) & (t < 2.5 * (col + 1))` is a Boolean mask: It is true for the samples of one 2.5-second column. `ax.set_aspect(0.4)` keeps the proportion of the paper, because 1 mV is drawn as 10 mm and 1 s as 25 mm, and 10 / 25 = 0.4.

In [ ]:
# The standard printout: 3 × 4 layout with a rhythm strip of lead II at 25 mm/s and 10 mm/mV
layout = [["I", "aVR", "V1", "V4"], ["II", "aVL", "V2", "V5"], ["III", "aVF", "V3", "V6"]]
shown = {"I": "I", "II": "II", "III": "III"}
ten = ptb.iloc[: int(10 * fs_ptb)]                               # the first 10 seconds
t = ten.index.to_numpy()
fig, axes = plt.subplots(4, 1, figsize=(12, 7.6))
for row, leads_row in enumerate(layout):
    for col, lead in enumerate(leads_row):
        part = (t >= 2.5 * col) & (t < 2.5 * (col + 1))         # each lead fills 2.5 s
        axes[row].plot(t[part], ten[names[lead]].to_numpy()[part], color="k", lw=0.8)
        axes[row].text(2.5 * col + 0.06, 1.62, shown.get(lead, lead), fontsize=9)
axes[3].plot(t, ten["ii"].to_numpy(), color="k", lw=0.8)         # rhythm strip
axes[3].text(0.06, 1.62, shown["II"], fontsize=9)
for ax in axes:
    ax.set_xlim(0, 10)
    ax.set_ylim(-1.5, 2.0)
    ax.set_xticks(np.arange(0, 10.001, 0.2))                      # large square: 0.2 s
    ax.set_xticks(np.arange(0, 10.001, 0.04), minor=True)         # small square: 0.04 s
    ax.set_yticks(np.arange(-1.5, 2.001, 0.5))                    # large square: 0.5 mV
    ax.set_yticks(np.arange(-1.5, 2.001, 0.1), minor=True)        # small square: 0.1 mV
    ax.grid(which="major", color="#E8A0A0", lw=0.7)
    ax.grid(which="minor", color="#F6DADA", lw=0.4)
    ax.tick_params(which="both", labelbottom=False, labelleft=False, length=0)
    ax.set_aspect(0.4)                                            # 1 mV = 10 mm and 1 s = 25 mm
axes[3].set_xlabel("10 s at 25 mm/s and 10 mm/mV")
plt.tight_layout()
plt.show()

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c246)*

In [ ]:
# Einthoven's and Goldberger's relations on real data
print("max |II - (I + III)| =", round(float(np.abs(ptb["ii"] - (ptb["i"] + ptb["iii"])).max()), 4), "mV")
print("max |aVR + aVL + aVF| =", round(float(np.abs(ptb["avr"] + ptb["avl"] + ptb["avf"]).max()), 4), "mV")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c247)*

In [ ]:
# «The measurement of «Stop 2 · Ready application» on the real record|«Durak 2 · Hazır uygulama» bölümündeki ölçüm, gerçek kayıtta»
leads_ptb = ptb[[names[k] for k in vclinic.LEADS]].to_numpy()
m_ptb = measure_ecg(leads_ptb, fs_ptb)
pd.Series(m_ptb).round(1)

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c248)*

**Observe and reflect.** The algorithm measures a QRS duration of about 137 ms in this record. Look at lead V1 and at the inferior leads in the figure. Which pattern could explain a QRS of this width, and what do the inferior leads show? The 2009 recommendations ask that computer measurements be checked by eye: Measure one beat yourself on the figure and compare.

**Advanced (optional).** The three Frank leads `vx`, `vy` and `vz` record the heart vector almost directly. Plot the QRS loop of one beat in three dimensions. Textbooks use different sign conventions for the Frank leads, so the loop is drawn here as recorded.

In [ ]:
# Advanced (optional): the QRS loop of one beat from the Frank leads
r_ptb = detect_r_peaks(ptb["ii"].to_numpy(), fs_ptb)
k = r_ptb[3]
loop = ptb[["vx", "vy", "vz"]].to_numpy()[k - int(0.06 * fs_ptb):k + int(0.08 * fs_ptb)]
fig = plt.figure(figsize=(5.5, 5))
ax = fig.add_subplot(projection="3d")
ax.plot(loop[:, 0], loop[:, 1], loop[:, 2], color="#9B1B1B")
ax.set_xlabel("vx (mV)")
ax.set_ylabel("vy (mV)")
ax.set_zlabel("vz (mV)")
plt.show()

#### Working with an AI assistant: The code safety checklist

*Lecture page: [Working with an AI assistant, part 1: The code safety checklist](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#ai-pair-programming-part-1-the-code-safety-checklist)*

An AI assistant can write most of the code of this course, and it can also write code that runs without error and returns a wrong number. The course therefore treats AI-written code like a new member of the team: It is welcome, and it is checked. The prompt card below gives the structure of a good request; it is the card of Prompt 3.4 in «Stop 3 · Prompt and application».

**Prompt card.** *Role:* You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there. *Context:* On an ECG sampled 500 times per second, the QT interval and the RR interval were measured as numbers of samples. The cardiologist wants the QT interval corrected by Bazett's method, in milliseconds. *Data:* The notebook holds a table named "data" with one row: The column "QT" gives the QT interval and the column "RR" the RR interval, both as numbers of samples. The recording was sampled 500 times per second. *Task:* Turn the QT interval into milliseconds and the RR interval into seconds. Correct the QT by Bazett's method: Divide the QT in milliseconds by the square root of the RR interval in seconds. *Constraints:* Use only this table. Do not assume the number of samples per second; use the 500 given here. The code must not read files, download anything from the internet or install programs. *Output:* Print the corrected QT in milliseconds with one decimal and keep it under the name "answer". *Test:* The same calculation must give exactly 400 ms for a QT of 200 samples and an RR interval of 500 samples, because the RR interval is then exactly 1 second.

**Check before using.** The lecture page gives the code safety checklist of the course: *sign in* before running (units, sampling frequency, lead, no patient identifier), *time out* while running (errors, physiological values) and *sign out* before using (comparison with a known truth, every line explained, prompt and changes logged).

An AI assistant produced the function below from a shorter prompt that gave no sampling frequency.

**Start cell.** For the section «Working with an AI assistant: The code safety checklist», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Working with an AI assistant: The code safety checklist»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

In [ ]:
# Code written by an AI assistant from a short prompt
def qtc_from_samples(qt_samples, rr_samples, fs=360):
    qt_ms = qt_samples / fs * 1000
    rr_ms = rr_samples / fs * 1000
    return qt_ms / np.sqrt(rr_ms)


print(qtc_from_samples(196, 353))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#nbref-c255)*

**Observe and reflect.** The function runs without an error. Go through the checklist: Which pause catches the result, and what are the two errors? One of them comes from a default value that the prompt never asked for.

**AI usage log.** Each weekly project adds a short log with one row per use of an AI assistant:

| Tool | Prompt (short) | What was checked | What was changed |
|---|---|---|---|
| (name of the assistant) | Bazett QTc from samples | Result 466.5 ms against the true values of VC-0317 | Removed the default fs; RR in seconds |

**The results of the prompts.** The cell below computes the results of the prompts of Stop 3 with the code of this application and checks them as the check cells of the prompts do.

In [ ]:
# The results of the prompts of this stop, computed in Python
from scipy.signal import find_peaks

n_atrial = ann_symbols.count("A")
print("Atrial premature beats in record 100:", n_atrial)
teach.check("Prompt 3.1", n_atrial)
beats = [(s, b) for s, b in zip(ann_samples, ann_symbols) if b != "+"]   # "+" marks a change of rhythm, not a beat
kinds = np.array([b for _, b in beats])
rr_all = np.diff([s for s, _ in beats]) / fs_mit * 1000
nn_all = rr_all[(kinds[:-1] == "N") & (kinds[1:] == "N")]
sdnn_all = round(np.std(nn_all, ddof=1), 1)
print("SDNN of the whole recording (ms):", sdnn_all)
teach.check("Prompt 3.2", sdnn_all)
lead_i = ptb["i"].to_numpy()
r_ptb, _ = find_peaks(lead_i, height=lead_i.max() / 2, distance=int(0.3 * fs_ptb))
hr_ptb = round(60 / np.diff(r_ptb / fs_ptb).mean(), 1)
print("Heart rate of the PTB record (bpm):", hr_ptb)
teach.check("Prompt 3.3", hr_ptb)
qtc_safe = round((196 / 500 * 1000) / (353 / 500) ** 0.5, 1)   # the corrected calculation of the code safety checklist
print("QTc by Bazett from 196 and 353 samples at 500 Hz (ms):", qtc_safe)
teach.check("Prompt 3.4", qtc_safe)

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#the-paper-of-the-week) and continue reading at «The paper of the week». The weekly project is at the end of this Jupyter notebook, and the section «Weekly project» of the lecture page describes it.

## Weekly project

*Lecture page: [Weekly project](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#weekly-project) · [README of Week 1, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-01/README.md#weekly-project)*

**The project.** A script that draws an ECG and computes its basic measurements (PR, QRS, QT) on its own. It starts from the function `measure_ecg` of «Stop 2 · Ready application», adds the frontal QRS axis, checks the measurements against the true values of ten patients, follows the heart rate of record 100 over thirty minutes and chooses between Bazett and Fridericia for each patient.

This part stands on its own: Its start cell loads the data of the project and prepares the tables of the steps. The project can be built with the prompts below, one for each step; each step works like a prompt of the stops, without a check cell or a ready code, because the results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log. Those who write Python find a start at the end of this part.

**Start cell.** For the weekly project, the cell below repeats the setup and the data calls of this part, so that the code runs without errors. The functions written in «Stop 2 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the weekly project
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

DATA_SOURCE = "web"   #@param ["library", "web", "computer"]
from vclinic import measure as _M
from vclinic.datasets import read_physionet_annotation
clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=DATA_SOURCE)
fs_mit = 360
project_patients = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                     "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]


def _project_measure(e):
    """Heart rate, PR, QRS, QT and frontal axis of one ECG, with the net QRS voltages of leads I and aVF in every beat."""
    L, fs = e.leads.to_numpy(), e.fs
    r = _M.detect_r_peaks(L[:, 1], fs)
    on, off = _M.qrs_bounds(L, fs, r)
    x20 = _M.lowpass(L[:, 1], fs, 20)
    sinus = e.truth["rhythm"] == "sinus"
    pr, qrs, qt, net_i, net_avf, t_prev = [], [], [], [], [], None
    for k in range(len(r) - 1):
        b = max(0, on[k] - int(0.03 * fs))
        te = _M.t_end_tangent(x20, fs, off[k], r[k + 1], np.median(x20[b:on[k]]))
        qrs.append((off[k] - on[k]) / fs * 1000)
        qt.append((te - on[k]) / fs * 1000)
        if sinus and t_prev is not None and np.isfinite(t_prev):
            pr.append((on[k] - _M.p_onset(x20, fs, on[k], earliest=t_prev + 0.02 * fs)) / fs * 1000)
        t_prev = te
        for col, out in ((0, net_i), (5, net_avf)):
            seg = L[on[k]:off[k] + 1, col] - np.median(L[b:on[k], col])
            out.append(seg.max() + seg.min())
    axis = [vclinic.frontal_axis(a, v) for a, v in zip(net_i, net_avf)]
    return {"hr": 60 / (np.mean(np.diff(r)) / fs), "pr": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
            "qrs": np.median(qrs), "qt": np.nanmedian(qt), "axis": float(np.median(axis)),
            "net_i": net_i, "net_avf": net_avf}


def _project_table(patients):
    rows = []
    for pid in patients:
        p = clinic.set_index("patient_id").loc[pid]
        e = vclinic.patient_ecg(clinic, pid)
        m, t = _project_measure(e), e.truth
        for name, key, true in [("heart rate", "hr", t["hr_bpm"]), ("PR", "pr", t["pr_ms"]), ("QRS", "qrs", t["qrs_ms"]),
                                ("QT", "qt", t["qt_ms"]), ("axis", "axis", t["axis_deg"])]:
            rows.append({"patient": pid, "rhythm": "sinus" if p["ecg_rhythm"] == "sinus" else "AF",
                         "conduction": p["ecg_conduction"], "measure": name,
                         "measured": round(float(m[key]), 1), "true": round(float(true), 1)})
    return pd.DataFrame(rows)


def _project_qt(patients):
    rows = []
    for pid in patients:
        p = clinic.set_index("patient_id").loc[pid]
        m = _project_measure(vclinic.patient_ecg(clinic, pid))
        rows.append({"patient": pid, "sex": "female" if p["sex"] == "F" else "male",
                     "heart rate": round(m["hr"], 1), "QT": round(m["qt"], 1)})
    return pd.DataFrame(rows)

# functions of earlier parts, kept unchanged in the course package
from vclinic.week01 import measure_ecg

### Project step 1: The frontal axis of the case

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to measure the frontal QRS axis of a 67-year-old woman from her resting ECG, beat by beat.  
> Data: The notebook holds a table named "data". Each row is a beat: The column "beat" gives its number, and the columns "I" and "aVF" give the net QRS voltage of these leads in millivolts. The net voltage is the largest positive plus the largest negative deflection of the QRS complex.  
> Task: For every beat, build the vector whose horizontal component is the voltage of lead I and whose vertical component is the voltage of lead aVF divided by 0.866. The frontal axis is the angle of this vector in degrees, measured from the direction of lead I, with positive angles towards the feet. Then take the median axis of all beats and print whether it lies in the normal range of −30 to +90 degrees.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the median axis in degrees with one decimal and keep it under the name "answer".  
> Test: A beat with 0.9 mV in lead I and −0.3 mV in lead aVF must give an axis of about −21 degrees. The registry gives an axis of 21 degrees for this patient, and the result should lie within about 10 degrees of it, because the net voltages also hold the small septal and terminal forces.  

In [ ]:
# Data of project step 1: the table "data" that the prompt describes
_m = _project_measure(vclinic.patient_ecg(clinic, "VC-0317"))
data = pd.DataFrame({"beat": np.arange(1, len(_m["net_i"]) + 1),
                     "I": np.round(_m["net_i"], 3), "aVF": np.round(_m["net_avf"], 3)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: Ten patients against the truth

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how well an automatic ECG measurement agrees with the true values in ten patients of the virtual clinic, among them patients with atrial fibrillation or left bundle branch block.  
> Data: The notebook holds a table named "data". Each row is one measurement of one patient: The column "patient" gives the number of the patient, "rhythm" says "sinus" or "AF", and "conduction" says "normal" or "LBBB" (left bundle branch block). The column "measure" names the measurement ("heart rate", "PR", "QRS", "QT" or "axis"), "measured" gives the value of the automatic measurement and "true" the true value. Heart rates are in beats per minute, intervals in milliseconds and axes in degrees. There is no PR interval in atrial fibrillation, so these cells are empty.  
> Task: Compute the difference, measured minus true, for every row. For each measurement, print the mean absolute difference and the largest absolute difference with the patient in whom it occurs. Then show all differences in one table, with one row per patient and one column per measurement.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the two summaries and the table, and keep the mean absolute difference of the QT interval in milliseconds under the name "answer".  
> Test: The table must have ten rows. For patient VC-0317, the differences of the PR, QRS and QT intervals must be only a few milliseconds.  

In [ ]:
# Data of project step 2: the table "data" that the prompt describes
# The ten patients of the project. The list may be changed; at least two of them need atrial fibrillation or a bundle branch block.
project_patients = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                     "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]
data = _project_table(project_patients)
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: The heart rate of record 100 over thirty minutes

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to see how the heart rate of a 69-year-old man changes over his 30-minute ambulatory ECG, record 100 of the MIT-BIH Arrhythmia Database.  
> Data: The notebook holds a table named "data". Each row is a mark that the cardiologists placed on the recording: The column "time" gives its place in seconds, and the column "label" its kind. "N" marks a normal beat, "A" an atrial premature beat and "V" a premature ventricular contraction; "+" marks a change of rhythm and is not a beat.  
> Task: Keep only the beats, that is, all labels except "+". For every beat after the first, compute the heart rate as 60 divided by the RR interval in seconds. Plot this heart rate against the time in minutes. Then compute the mean heart rate of the whole recording as 60 divided by the mean RR interval.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the mean heart rate in beats per minute with one decimal and keep it under the name "answer". Show the plot as well.  
> Test: An RR interval of exactly 1 second must give 60 beats per minute, and the mean heart rate must lie between 60 and 90 beats per minute.  

In [ ]:
# Data of project step 3: the table "data" that the prompt describes
data = pd.DataFrame({"time": np.asarray(ann_samples) / fs_mit, "label": ann_symbols})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: Bazett or Fridericia for each patient

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: For ten patients of the virtual clinic, the cardiologist wants to know whether the choice between Bazett's and Fridericia's correction changes the decision that the QT interval is prolonged.  
> Data: The notebook holds a table named "data". Each row is a patient: The column "patient" gives the number of the patient, and "sex" says "female" or "male". The column "heart rate" gives the measured heart rate in beats per minute, and "QT" the measured QT interval in milliseconds.  
> Task: For every patient, compute the RR interval in seconds as 60 divided by the heart rate. Compute the QTc by Bazett's formula, the QT divided by the square root of RR, and by Fridericia's formula, the QT divided by the cube root of RR. Call a QTc prolonged when it is 460 ms or longer in a woman, or 450 ms or longer in a man. List the patients for whom the two formulas lead to different decisions.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print a table with the two corrections and the two decisions, and the list of the disagreements. Keep the number of patients with different decisions under the name "answer".  
> Test: Patient VC-0317, a woman at about 85 beats per minute, must be prolonged by Bazett's formula and not by Fridericia's.  

In [ ]:
# Data of project step 4: the table "data" that the prompt describes
data = _project_qt(project_patients)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### A start in Python: another patient, another lead

The cell below measures one patient of the virtual clinic with `measure_ecg` and compares the result with the true values. Run the start cell above it first. Then change `PATIENT` and `LEAD` and run the cell again. Patients worth a look: VC-0183 (a long QT interval), VC-0018 (left bundle branch block), VC-0207 (a PR interval of 218 ms in the registry) and any patient whose `ecg_rhythm` is `AF`. `clinic[clinic["stemi_territory"] != "none"]` lists the patients with an acute ST-elevation myocardial infarction (STEMI), the topic of Week 5.

**Start cell.** For the start in Python, the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 2 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the start in Python
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=1, language="en", packages=["wfdb", "neurokit2"])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026

# functions of earlier parts, kept unchanged in the course package
from vclinic.week01 import measure_ecg

# repeated from the earlier cells: the data and the code that the cells below use
clinic = vclinic.make_cohort(n=500, seed=2026)   # the virtual clinic, as in Stop 1

In [ ]:
PATIENT = "VC-0183"
LEAD = "II"

e = vclinic.patient_ecg(clinic, PATIENT)
row = clinic[clinic["patient_id"] == PATIENT].iloc[0]
print(PATIENT, row["age"], row["sex"], row["main_diagnosis"], row["ecg_rhythm"], row["ecg_conduction"])
m = measure_ecg(e.leads.to_numpy(), e.fs, lead=LEAD, sinus=e.truth["rhythm"] == "sinus")
keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
display(pd.DataFrame({"measured": [m[k] for k in keys],
                      "true": [e.truth[k] for k in keys]}, index=keys).round(1))
fig = teach.plot_12lead(e.time, e.leads, title=PATIENT)
plt.show()

**End of the weekly project.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lecture.html#weekly-project): The section «Weekly project» describes the report and its submission.

## Application challenges by field

| Field | Challenge |
|---|---|
| Electrophysiology | Generate the ECG of VC-0183 at heart rates from 50 to 110 bpm (change `hr_bpm` in `e.params` and keep `qt_ms` scaled by $RR^{1/3}$). Plot QTc by Bazett and by Fridericia against heart rate and decide which one to report in a QT monitoring protocol. |
| Interventional cardiology | List the patients with a STEMI territory, draw their 12-lead ECGs and name the leads with ST elevation for each territory. Compare with the culprit artery that the territory suggests. |
| Heart failure | Compare SDNN and RMSSD of 5-minute recordings between the HFrEF patients and the patients without heart disease of the clinic. |
| Cardio-oncology | Build a QTc follow-up table for one patient on a QT-prolonging treatment, with serial ECGs at different heart rates, and flag the visits at which the QTc crosses the threshold of your local protocol. |
| Pediatric cardiology | The 2009 recommendations use lower QRS thresholds in children. Change the threshold of a wide QRS in a function of your own and discuss what else would change in a child's ECG. |
| Sports cardiology | Generate a resting ECG with a heart rate of 45 bpm and compare QTc by both formulas: Which one could label an athlete as abnormal? |
| Intensive care | Generate 5 minutes with `noise="clinic"` and measure Se and PPV of the R-peak detector against the true beat times. |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-01/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 1, described on the [week page](README.md), starts from the function `measure_ecg` of «Stop 2 · Ready application».

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>